### Package versions

Run the cell below to record the exact package versions used to generate the results in this notebook.


In [ ]:
# --- Environment / package versions (added for reproducibility) ---
import sys
try:
    from importlib.metadata import version, PackageNotFoundError
except ImportError:  # Python < 3.8
    from importlib_metadata import version, PackageNotFoundError
print("Python:", sys.version.split()[0])
for _pkg in ["numpy", "pandas", "scipy", "statsmodels", "matplotlib", "seaborn", "gseapy", "pyliftover", "openpyxl", "requests", "pyBigWig", "pyreadr", "adjustText", "pypdf"]:
    try:
        print(f"{_pkg}: {version(_pkg)}")
    except PackageNotFoundError:
        pass


# SLC38A4 locus — transcript-isoform CAGE scan, LD expansion and cCRE Registry V4 intersection

Two related analyses at the *SLC38A4* locus, consolidated into one notebook because they share the same liver chromatin-accessibility, H3K27ac and ENCODE Registry V4 evidence files (Section 1 below), so both stay consistent with a single source of truth rather than two notebooks that could drift apart.

**Part A** (Supplementary Methods 5.1) scans the locus for FANTOM5 CAGE transcription-start-site clusters, independent of any pre-specified candidate, to establish which *SLC38A4* GENCODE transcript isoform is expressed in liver and hepatocytes (Table S13).

**Part B** (Supplementary Methods 5.3) starts from the lead GWAS variant, expands to its linkage-disequilibrium proxies, intersects them with the **ENCODE Registry of candidate cis-regulatory elements V4** (Nature 2026,
doi:10.1038/s41586-025-09909-9 — 2.37 million human cCREs, up from 0.9 million in V3), and
restricts the result to elements with evidence of activity in liver. The output is the set of
proxies satisfying all three criteria; no candidate is specified in advance.

## Design

| Step | Source | Role |
|---|---|---|
| LD expansion | **LDlink** `LDproxy`, 1000 Genomes **GBR** | primary source; broadest variant coverage |
| LD expansion | **Ensembl REST** `/ld/human`, `1000GENOMES:phase_3:GBR` | independent estimate; also contributes proxies LDlink does not return |
| cCRE V4 | **UCSC REST API**, hg38 | track auto-discovered and its label recorded, so the version used is self-documenting |
| Intersection | **`bedtools intersect`** (required), verified against an independent interval calculation | the two must agree |
| Liver activity | per-biosample cCRE classifications (SCREEN / ENCODE portal) | restricts the shortlist to cCREs active in liver tissue, hepatocytes or other primary liver cells |

Two LD sources are queried because a proxy set is only as good as its LD estimates. Where a
proxy sits close to the R² threshold the inclusion decision can be sensitive to the reference
panel and to the service used, so both estimates are reported for every proxy and any variant
whose threshold membership depends on the source is flagged rather than silently resolved.

## Everything runs on GRCh38

LD is queried with `genome_build=grch38` and cCREs are hg38, so there is **construction.

## Prerequisites

- **LDlink personal access token** — free, register at
  https://ldlink.nih.gov/apiaccess, then set `LDLINK_TOKEN` below. Without it the
  notebook still runs on the Ensembl LD source alone and says so.
- Outbound HTTPS to `ldlink.nih.gov`, `rest.ensembl.org`, `api.genome.ucsc.edu`.
  **Cannot run in a network-restricted sandbox — run locally.**

## Outputs

- `SLC38A4_LD_proxies_GRCh38.csv` — every proxy with R², D′, distance, alleles, both LD sources
- `SLC38A4_proxies_cCRE_V4.csv` — proxies annotated with overlapping cCRE accession and class
- `SLC38A4_cCRE_prioritisation.csv` — the shortlist: proxies above the R² threshold that fall in a cCRE
- `SLC38A4_proxies_summary.csv` — **one row per proxy**: R² against the lead variant plus an `overlaps_ccre` True/False column
- `SLC38A4_LD_proxies.bed` — all proxies as a browser track (hg38), R²-scored and colour-coded
- `SLC38A4_cCRE_prioritisation.bed` — the prioritisation set (R² ≥ threshold, in a cCRE, **and liver-active**), coloured by cCRE class
- `SLC38A4_cCRE_notLiverActive.bed` — variants excluded solely by the liver filter, kept for transparency
- `SLC38A4_cCREs_V4_window.bed` — the cCREs themselves, for the Figure 5A view
- `SLC38A4_cCRE_provenance.json` — versions, track label, thresholds, timestamps


## 1. Setup

In [ ]:
import hashlib, json, os, sys, time, platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# ----------------------------------------------------------------- configuration
LEAD_RSID      = "rs11183620"       # lead GWAS variant at the SLC38A4 locus - the only
                                    # variant specified in advance
POP            = "GBR"              # 1000 Genomes reference population
R2_THRESHOLD   = 0.7                # proxy inclusion threshold
WINDOW_KB      = 250                # +/- this distance from the lead variant
GENOME         = "hg38"

# Read from the environment - never commit a token to a repository.
# export LDLINK_TOKEN=your_token        (then start Jupyter from that shell)
LDLINK_TOKEN   = os.environ.get("LDLINK_TOKEN")
CONTACT_EMAIL  = "i.dos-santos-cebola@imperial.ac.uk"

LDLINK   = "https://ldlink.nih.gov/LDlinkRest"
ENSEMBL  = "https://rest.ensembl.org"
UCSC     = "https://api.genome.ucsc.edu"
OUTDIR   = Path(".")

# bedtools is required. Leave as None to search PATH, or give the full path if the
# kernel's PATH does not include it (common when Jupyter is started from a launcher
# rather than a terminal - e.g. "/opt/homebrew/bin/bedtools").
BEDTOOLS_BIN = None

# Liver cCRE inputs (ENCODE Registry V4 per-biosample files, subset to chr12q).
# Defined here rather than in section 9 so that section 8 can use them too.
LIVER_CCRE_DIR  = Path("liver_cCREs/chr12q")        # per-biosample chr12q subsets
LIVER_MANIFEST  = Path("liver_cCREs/manifest.tsv")  # biosample / category / assay coverage
# Pooled liver track. Only liver.all is used: it aggregates signal over all liver
# biosamples and is better powered than the individual samples, many of which have
# only one assay. (A liver.noccl aggregate excluding cancer lines also exists in
# SCREEN but is not used here.)
LIVER_AGGREGATE = {"liver.all": LIVER_CCRE_DIR / "liver.all.cCREs.chr12q.bed"}

# ---------------------------------------------------------------- evidence panels
# Liver regulatory evidence, used as the prioritisation criteria (sections 9b, 10).
# Each panel has per-sample peak calls plus a replicated set.
#
# DATA SOURCE - PHH: 6 adult donors, Omni-ATAC, published in Jamialahmadi et al.,
# Clin Mol Hepatol 2025;31(4):1252-1268 (doi:10.3350/cmh.2025.0159), EGA
# EGAD50000001751. The peak files here are a later re-analysis: that paper called
# peaks with MACS2 --broad -q 0.01, whereas these are narrow calls (donors at
# -p 0.05, pooled at -q 0.01, re-run 16/05/2025). The replication rule is unchanged.
#
# DATA SOURCE - ENCODE liver ATAC and H3K27ac: ENCODE experiments re-processed
# in-house; these are NOT ENCODE's published peak calls.
ATAC_ROOT = Path("liver_cCREs/CebolaLab_CREs")
EVIDENCE_PANELS = {
    "PHH": dict(
        role="accessibility",
        dir=ATAC_ROOT / "PHH_CebolaLab",
        donor_glob="PHH*-C_peaks.narrowPeak",
        replicated="replicated_peaks.bed",          # pooled peak in >=3 of 6 donors
        label="primary human hepatocyte ATAC-seq (6 adult donors)"),
    "ENCODE_liver": dict(
        role="accessibility",
        dir=ATAC_ROOT / "ENCODE_Liver/ATAC",
        donor_glob="ENCSR*_peaks_SORTED.bed",
        replicated="Liver_ATAC_replicated_peaks.bed",   # >=3 of 6, 50% overlap
        label="whole-liver ATAC-seq (6 ENCODE experiments, peaks called in-house)"),
    "H3K27ac": dict(
        role="activity",
        dir=ATAC_ROOT / "ENCODE_Liver/H3K27ac",
        donor_glob="*_pooled_macs2_peaks.broadPeak",
        replicated="replicated_peaks_in_2.bed",         # peak in >=2 of 4 donors
        label="liver H3K27ac ChIP-seq (4 ENCODE experiments, peaks called in-house)"),
}

# ------------------------------------------------------------ prioritisation rule
# Tier 1 - the CRE core: replicated ATAC peaks from CORE_PANEL define the element.
#          A variant outside the core is not a candidate, however strong the
#          surrounding signal - a variant in the shoulder of a peak is not in the
#          element. The hepatocyte panel defines the core because SLC38A4 is
#          expressed in hepatocytes; whole-liver ATAC peaks are broader and are
#          reported as corroboration only (CORROBORATING_PANELS).
# Tier 2 - activity: H3K27ac, evaluated ONLY for variants inside the core.
# Tier 3 - Registry V4 membership and class: annotation, never a gate. The
#          Registry funnel is reported alongside in section 10 for comparison.
CORE_PANEL           = "PHH"
CORE_MARK_PANELS     = ("H3K27ac",)
CORROBORATING_PANELS = ("ENCODE_liver",)

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": f"SLC38A4-cCRE-prioritisation ({CONTACT_EMAIL})"})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)


# ------------------------------------------------------------------- HTTP helper
# Checks status and Content-Type BEFORE parsing, and on failure shows what actually
# came back - which is how you distinguish a rate limit from an institutional proxy
# page from a real outage.
class ApiError(RuntimeError):
    pass

def _describe(r):
    body = (r.text or "")[:300].replace("\n", " ")
    return (f"HTTP {r.status_code}, Content-Type={r.headers.get('Content-Type','?')}, "
            f"body[:300]={body!r}")

def _request(method, url, *, retries=4, timeout=90, want="json", **kw):
    last = None
    for attempt in range(retries):
        try:
            r = SESSION.request(method, url, timeout=timeout, **kw)
        except requests.RequestException as e:
            last = f"{type(e).__name__}: {str(e)[:150]}"
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 2 ** attempt))
            print(f"    rate limited, sleeping {wait}s"); time.sleep(wait)
            last = _describe(r); continue
        if r.status_code in (500, 502, 503, 504):
            last = _describe(r)
            html = "html" in (r.headers.get("Content-Type") or "").lower()
            if html and attempt >= 1:
                raise ApiError(
                    f"{method} {url}\n    {last}\n"
                    "    A 5xx with an HTML body twice running usually means the URL or\n"
                    "    payload is wrong for this service, not a transient outage. Check\n"
                    "    the endpoint against current documentation before retrying.")
            time.sleep(2 ** attempt); continue
        if r.status_code != 200:
            raise ApiError(f"{method} {url}\n    {_describe(r)}")
        if want == "text":
            return r.text
        if "json" not in (r.headers.get("Content-Type") or "").lower():
            raise ApiError(f"{method} {url} returned non-JSON\n    {_describe(r)}\n"
                           "    If the body looks like HTML, a proxy or captive portal is "
                           "intercepting the request - try off the institutional network.")
        try:
            return r.json()
        except ValueError as e:
            raise ApiError(f"{method} {url} JSON parse failed: {e}\n    {_describe(r)}")
    raise ApiError(f"{method} {url} failed after {retries} attempts. Last: {last}")

def get_json(url, **kw): return _request("GET", url, want="json", **kw)
def get_text(url, **kw): return _request("GET", url, want="text", **kw)
def post_json(url, **kw):     return _request("POST", url, want="json", **kw)


# -------------------------------------------------------------- frame schemas
# A parser that branches can silently fail to create a column when no row
# populates it. Declaring the schema and reindexing makes columns always present.
LD_COLS = ["rsid", "chrom", "pos_grch38", "alleles", "maf", "distance_bp",
           "r2_ldlink", "dprime_ldlink", "r2_ensembl", "dprime_ensembl"]
CCRE_COLS = ["ccre_accession", "chrom", "start", "end", "ccre_class", "ccre_label"]

def frame(records, columns, name=""):
    df = pd.DataFrame(list(records)).reindex(columns=columns)
    for c in [c for c in df.columns if c.startswith(("r2_", "dprime_", "maf"))]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in [c for c in df.columns if c in ("pos_grch38", "start", "end", "distance_bp")]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    empty = [c for c in columns if df[c].isna().all()]
    if empty and name:
        print(f"  note: {name} has no values for {empty}")
    return df

print("python  ", sys.version.split()[0])
print("pandas  ", pd.__version__)
print("platform", platform.platform())
print("LDlink token set:", bool(LDLINK_TOKEN))
if not LDLINK_TOKEN:
    print("  -> export LDLINK_TOKEN=... and restart the kernel;"
          " request one free at https://ldlink.nih.gov/apiaccess")


### 1b. Preflight connectivity check

Run this first. It reports what each endpoint actually returns, so a failure names the
problem instead of surfacing as a parse error ten cells later.

Ensembl and UCSC are required. LDlink is optional but strongly preferred: it returns
substantially more proxies.


In [ ]:
CHECKS = [
    ("Ensembl ping", f"{ENSEMBL}/info/ping", {"content-type": "application/json"}, True),
    ("UCSC    api",  f"{UCSC}/list/ucscGenomes", None, True),
    ("LDlink  api",  f"{LDLINK}/ldproxy",
        {"var": LEAD_RSID, "pop": POP, "r2_d": "r2", "window": 10000,
         "genome_build": "grch38", "token": LDLINK_TOKEN or "MISSING"}, False),
]

ok = True
for label, url, params, required in CHECKS:
    try:
        r = SESSION.get(url, params=params, timeout=(10, 30))
        ct = (r.headers.get("Content-Type") or "?").split(";")[0]
        verdict = "OK" if r.status_code == 200 else "FAIL"
        note = ""
        if r.status_code == 200 and "error" in (r.text or "")[:200].lower():
            verdict, note = "FAIL", "  <- endpoint returned an error message"
        if r.status_code != 200:
            note = "  " + (r.text or "")[:120].replace("\n", " ")
        print(f"{verdict:4s} {label:14s} HTTP {r.status_code}  {ct}{note}")
        if required and verdict != "OK":
            ok = False
    except Exception as e:
        print(f"FAIL {label:14s} {type(e).__name__}: {str(e)[:110]}")
        if required:
            ok = False

print()
print("Required endpoints reachable - proceed." if ok else
      "A REQUIRED endpoint is unreachable. Do not interpret partial results.\n"
      "  non-JSON/HTML body -> institutional proxy; try off-VPN\n"
      "  HTTP 429           -> rate limited, wait\n"
      "  connect timeout    -> firewall blocking outbound HTTPS")
if not LDLINK_TOKEN:
    print("\nNOTE: no LDlink token. The notebook will run on the Ensembl LD source only,\n"
          "      which covers fewer variants. Register (free) at\n"
          "      https://ldlink.nih.gov/?tab=apiaccess for the full proxy set.")


## Part A — SLC38A4 transcript-isoform CAGE scan (Supplementary Methods 5.1)

FANTOM5 CAGE (CTSS) data from three primary human hepatocyte donors and one adult liver
pool are scanned across the *SLC38A4* locus for clusters of capped 5' ends, independent of
any pre-specified candidate. Each cluster found is then annotated against GENCODE transcript
models (fetched live from Ensembl) and against the same liver chromatin accessibility /
H3K27ac / ENCODE Registry V4 evidence used in Part B below, so a cluster's support can be
read off directly rather than asserted.

| Step | What | Why |
|---|---|---|
| Cluster calling | merge CTSS positions <=20 bp apart, per sample, per strand | a CAGE tag cluster, not a single base |
| Consensus | union of per-sample clusters | a locus-wide, sample-independent set of candidate TSS regions |
| Reproducibility | >=2 of 3 hepatocyte donors at >=0.5 TPM | the liver pool is a pool, not a biological replicate, so it is reported but not required |
| GENCODE overlay | live Ensembl REST fetch, `ENSG00000139209` | assigns each cluster to a transcript model (or flags it as unannotated) without hardcoding a snapshot |
| Chromatin support | `bedtools intersect` against the same PHH ATAC, ENCODE liver ATAC, H3K27ac and Registry V4 files configured in Section 1 | reuses the exact evidence already validated for this locus in Part B, rather than a second copy |

No candidate is specified in advance: the scan runs across the whole locus and every
consensus cluster with any reproducible signal is reported in the output table (Table S13),
not just the ones that end up looking interesting.

**Caveats, stated up front**

- CAGE tags mark 5'-capped RNA ends. That includes genuine transcription start sites, but
  also capped degradation and nonsense-mediated-decay intermediates - a cluster is necessary
  but not sufficient evidence of a stable, functional promoter.
- These four libraries are resting-state hepatocytes/liver. If an alternative promoter is used
  only under the inflammatory conditions of SAH, this data would not be expected to show it.
- The transcript models fetched here are annotated by GENCODE's TAGENE pipeline
  (`source: havana_tagene`), which itself incorporates CAGE-type evidence. Agreement between a
  cluster here and a TAGENE transcript start is one line of evidence among several, not fully
  independent confirmation.
- Several GENCODE transcripts for this gene share an essentially identical annotated TSS
  (see the "extended (5' of canonical)" family below); where a CAGE cluster's nearest transcript
  is tied on distance between two or more such transcripts, which one is reported as
  `nearest_transcript_id` can depend on the order Ensembl returns them in, but the reported
  category and distance do not change.


### A1. Configuration

In [ ]:
import gzip
from pathlib import Path
import numpy as np

# FANTOM5 CAGE (CTSS) source files -- three primary human hepatocyte donors + one adult
# liver pool. HEP_DONORS are true biological replicates; the liver pool is a pool of many
# donors, reported for context but not counted towards the reproducibility rule below.
CAGE_DIR = Path("liver_cCREs/CAGE")
CAGE_FILES = {
    "Hepatocyte_donor1": "Hepatocyte%2c%20donor1.CNhs12340.11523-119I2.hg38.nobarcode.ctss.bed.gz",
    "Hepatocyte_donor2": "Hepatocyte%2c%20donor2.CNhs12349.11603-120I1.hg38.nobarcode.ctss.bed.gz",
    "Hepatocyte_donor3": "Hepatocyte%2c%20donor3.CNhs12626.11684-122I1.hg38.nobarcode.ctss.bed.gz",
    "Liver_adult_pool1":  "liver%2c%20adult%2c%20pool1.CNhs10624.10018-101C9.hg38.nobarcode.ctss.bed.gz",
}
HEP_DONORS = ["Hepatocyte_donor1", "Hepatocyte_donor2", "Hepatocyte_donor3"]

# Named CAGE_CHROM/CAGE_WINDOW (rather than CHROM/WINDOW) because this scan's window is
# independent of, and much narrower than, the +/-WINDOW_KB LD window used in Part B below.
CAGE_CHROM = "chr12"
# SLC38A4 gene envelope (all GENCODE transcripts, ENSG00000139209) is
# chr12:46,763,642-46,832,500 (GRCh38). Padded +/-20 kb so an alternative TSS just outside
# the annotated envelope would not be missed.
CAGE_WINDOW = (46_740_000, 46_855_000)

CLUSTER_GAP_BP  = 20    # max bp gap between consecutive CTSS positions merged into one
                        # cluster (per sample, per strand)
MIN_CLUSTER_TPM = 0.5   # per-sample calling threshold for "detected"
MIN_HEP_DONORS  = 2     # of 3, at >=MIN_CLUSTER_TPM, to call a cluster reproducible

# GENCODE overlay -- reuses the ENSEMBL endpoint already configured in Section 1 above.
GENE_ID = "ENSG00000139209"                    # SLC38A4
CANONICAL_TRANSCRIPT_ID = "ENST00000266579"    # MANE Select

# Chromatin support -- reuses EVIDENCE_PANELS and LIVER_AGGREGATE from Section 1 above
# (same PHH ATAC, ENCODE liver ATAC, H3K27ac and Registry V4 files as Part B), rather than
# a second, independently-maintained copy of these paths.
CHROMATIN_PANELS = {
    "PHH_ATAC_replicated":          Path(EVIDENCE_PANELS["PHH"]["dir"]) / EVIDENCE_PANELS["PHH"]["replicated"],
    "ENCODE_liver_ATAC_replicated": Path(EVIDENCE_PANELS["ENCODE_liver"]["dir"]) / EVIDENCE_PANELS["ENCODE_liver"]["replicated"],
    "H3K27ac_replicated_in2":       Path(EVIDENCE_PANELS["H3K27ac"]["dir"]) / EVIDENCE_PANELS["H3K27ac"]["replicated"],
}
LIVER_CCRE_BED = LIVER_AGGREGATE["liver.all"]

# GWAS context, for the locus plot only -- not used anywhere in the scan or the calling rules.
RS4994910_POS = 46_831_288

print(f"Scanning {CAGE_CHROM}:{CAGE_WINDOW[0]:,}-{CAGE_WINDOW[1]:,} "
      f"({CAGE_WINDOW[1]-CAGE_WINDOW[0]:,} bp)")
print("Chromatin panels (shared with Part B):")
for k, v in CHROMATIN_PANELS.items():
    print(f"  {k:32s} {v}")
print(f"  {'Registry V4 (liver.all)':32s} {LIVER_CCRE_BED}")


### A2. Parse CTSS files

One row per FANTOM5 library per observed CAGE tag position (`.ctss.bed.gz`, hg38, 5-column
BED + strand). Library size is the sum of tag counts across the *whole* file (not just this
window), so that per-cluster values can be expressed as tags per million (TPM) and compared
fairly across libraries of very different depth.

In [ ]:
import pandas as pd

def read_ctss(path, chrom=None, lo=None, hi=None):
    """Return (windowed_rows, library_size). windowed_rows is a list of
    (pos, count, strand) tuples restricted to chrom:lo-hi; library_size counts every
    tag in the file regardless of chromosome or window."""
    rows, total = [], 0
    with gzip.open(path, "rt") as fh:
        for line in fh:
            p = line.rstrip("\n").split("\t")
            c = int(p[4])
            total += c
            if chrom is not None and p[0] == chrom and lo <= int(p[1]) <= hi:
                rows.append((int(p[1]), c, p[5]))
    return rows, total

libsize = {}
long_rows = []
for label, fname in CAGE_FILES.items():
    fp = CAGE_DIR / fname
    if not fp.exists():
        raise FileNotFoundError(f"Missing CAGE file for {label!r}: {fp}")
    rows, total = read_ctss(fp, CAGE_CHROM, *CAGE_WINDOW)
    libsize[label] = total
    for pos, count, strand in rows:
        long_rows.append((label, pos, count, strand))

ctss = pd.DataFrame(long_rows, columns=["sample", "pos", "count", "strand"])
print("Library sizes (total tags, whole file):")
for k, v in libsize.items():
    print(f"  {k:20s} {v:>10,}")
print(f"\nCTSS positions with >=1 tag in the window: {len(ctss):,}")


### A3. Per-sample cluster calling

Within each sample and strand, consecutive CTSS positions no more than `CLUSTER_GAP_BP`
apart are merged into one cluster. This is a lightweight, transparent stand-in for
CAGEr/paraclu decomposition - appropriate here because there are only four libraries and the
question is simply "where are there clusters, and how big are they", not tag-level shape
decomposition.

In [ ]:
def call_clusters(df_sample_strand, gap):
    d = df_sample_strand.sort_values("pos")
    clusters, cur, last = [], [], None
    for _, r in d.iterrows():
        if last is not None and r["pos"] - last > gap:
            clusters.append(cur); cur = []
        cur.append((r["pos"], r["count"]))
        last = r["pos"]
    if cur:
        clusters.append(cur)
    out = []
    for cl in clusters:
        poss, cnts = zip(*cl)
        out.append({
            "start": min(poss), "end": max(poss),
            "summit": poss[int(np.argmax(cnts))],
            "total_count": sum(cnts), "n_positions": len(poss),
        })
    return out

per_sample = []
for label in CAGE_FILES:
    for strand in ["+", "-"]:
        sub = ctss[(ctss["sample"] == label) & (ctss["strand"] == strand)]
        if sub.empty:
            continue
        for cl in call_clusters(sub, CLUSTER_GAP_BP):
            cl.update(sample=label, strand=strand, tpm=cl["total_count"] / libsize[label] * 1e6)
            per_sample.append(cl)

per_sample = pd.DataFrame(per_sample)
print(f"Per-sample clusters called: {len(per_sample)}")
per_sample.sort_values("tpm", ascending=False).head(10)


### A4. Consensus clusters and reproducibility

Per-sample cluster intervals are pooled per strand and merged (same gap rule) into consensus
regions, independent of which sample(s) contributed them. Each hepatocyte donor and the liver
pool are then re-quantified inside every consensus region - so a region contributed by only
one sample still gets an honest (possibly zero) value for the other three, rather than being
scored only where it was first detected.

In [ ]:
def merge_intervals(intervals, gap):
    merged = []
    for s, e in sorted(intervals):
        if merged and s - merged[-1][1] <= gap:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

consensus_rows = []
for strand in ["+", "-"]:
    ivs = list(zip(per_sample.loc[per_sample.strand == strand, "start"],
                    per_sample.loc[per_sample.strand == strand, "end"]))
    for s, e in merge_intervals(ivs, CLUSTER_GAP_BP):
        consensus_rows.append({"strand": strand, "start": s, "end": e})

results = []
for c in consensus_rows:
    sub = ctss[(ctss.strand == c["strand"]) & (ctss.pos >= c["start"]) & (ctss.pos <= c["end"])]
    if sub.empty:
        continue
    rec = {"strand": c["strand"], "start": c["start"], "end": c["end"],
           "width_bp": c["end"] - c["start"] + 1}
    pooled = sub.groupby("pos")["count"].sum()
    rec["summit"] = int(pooled.idxmax())
    for label in CAGE_FILES:
        rec[f"tpm_{label}"] = sub.loc[sub["sample"] == label, "count"].sum() / libsize[label] * 1e6
    rec["tpm_max"] = max(rec[f"tpm_{l}"] for l in CAGE_FILES)
    rec["n_hep_donors_detected"] = sum(rec[f"tpm_{d}"] >= MIN_CLUSTER_TPM for d in HEP_DONORS)
    rec["reproducible"] = rec["n_hep_donors_detected"] >= MIN_HEP_DONORS
    results.append(rec)

clusters = pd.DataFrame(results).sort_values("tpm_max", ascending=False).reset_index(drop=True)
print(f"Consensus clusters in window: {len(clusters)}")
print(f"Reproducible (>={MIN_HEP_DONORS}/3 hepatocyte donors >={MIN_CLUSTER_TPM} TPM): "
      f"{clusters['reproducible'].sum()}")
clusters.head(15)


### A5. GENCODE transcript overlay (live Ensembl fetch)

Every transcript GENCODE assigns to *SLC38A4* is pulled directly from Ensembl (not a
hardcoded snapshot; the same `ENSEMBL` endpoint configured in Section 1), and each
reproducible cluster is matched to the nearest annotated transcript start within 150 bp, with
the distance and the transcript's biotype/length category reported rather than assumed.

In [ ]:
def fetch_transcripts(gene_id):
    r = requests.get(f"{ENSEMBL}/overlap/id/{gene_id}",
                      params={"feature": "transcript"},
                      headers={"Content-Type": "application/json"}, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json())

tx = fetch_transcripts(GENE_ID)
tx = tx[tx["Parent"] == GENE_ID].copy()          # this gene only, not overlapping neighbours
tx["tss"] = np.where(tx["strand"] == -1, tx["end"], tx["start"])  # 5' end per strand
canon_tss = int(tx.loc[tx.transcript_id == CANONICAL_TRANSCRIPT_ID, "tss"].iloc[0])

def categorise(row):
    if row.biotype == "nonsense_mediated_decay":
        return "NMD"
    if row.transcript_id == CANONICAL_TRANSCRIPT_ID:
        return "canonical (MANE Select)"
    if row.biotype != "protein_coding":
        return row.biotype
    delta = (row.tss - canon_tss) if row.strand == -1 else (canon_tss - row.tss)
    if delta > 50:
        return "extended (5' of canonical)"
    if delta < -50:
        return "internal/shorter than canonical"
    return "canonical-adjacent"

tx["category"] = tx.apply(categorise, axis=1)
print(f"{len(tx)} transcripts annotated for {GENE_ID}; canonical TSS = chr12:{canon_tss:,}")
tx.groupby("category")["transcript_id"].nunique().sort_values(ascending=False)


In [ ]:
def nearest_transcript(summit, strand_sign, max_dist=150):
    cand = tx[tx.strand == strand_sign].copy()
    if cand.empty:
        return None
    cand["dist"] = (cand["tss"] - summit).abs()
    best = cand.sort_values("dist").iloc[0]
    if best["dist"] > max_dist:
        return None
    return best

strand_to_sign = {"+": 1, "-": -1}
match_cols = {"nearest_transcript_id": [], "nearest_transcript_name": [],
              "nearest_transcript_category": [], "dist_to_nearest_tx_tss_bp": []}
for _, row in clusters.iterrows():
    m = nearest_transcript(row["summit"], strand_to_sign[row["strand"]])
    if m is None:
        match_cols["nearest_transcript_id"].append(None)
        match_cols["nearest_transcript_name"].append(None)
        match_cols["nearest_transcript_category"].append("unannotated")
        match_cols["dist_to_nearest_tx_tss_bp"].append(None)
    else:
        match_cols["nearest_transcript_id"].append(m["transcript_id"])
        match_cols["nearest_transcript_name"].append(m["external_name"])
        match_cols["nearest_transcript_category"].append(m["category"])
        match_cols["dist_to_nearest_tx_tss_bp"].append(int(m["dist"]))
for k, v in match_cols.items():
    clusters[k] = v

clusters.loc[clusters.reproducible].sort_values("tpm_max", ascending=False)[
    ["strand", "summit", "tpm_max", "n_hep_donors_detected",
     "nearest_transcript_category", "dist_to_nearest_tx_tss_bp"]
]


### A6. Chromatin support

Each reproducible cluster is intersected against the same accessibility, H3K27ac and ENCODE
Registry V4 files configured in Section 1 (`CHROMATIN_PANELS`, `LIVER_CCRE_BED`) - `bedtools
intersect` is required, not a pandas re-implementation, for consistency with the rest of this
notebook. This is the first use of bedtools in this notebook, so `BEDTOOLS` is resolved here
(Part B's own cross-check cell later just confirms the same executable).

In [ ]:
import shutil, subprocess

BEDTOOLS = BEDTOOLS_BIN or shutil.which("bedtools")
if not BEDTOOLS:
    raise RuntimeError(
        "bedtools not found.\n"
        "  Install:  conda install -c conda-forge -c bioconda bedtools\n"
        "            (on Apple Silicon, where bioconda has no osx-arm64 build:\n"
        "             brew install bedtools)\n"
        "  If installed but not on this kernel's PATH, check with  !bedtools --version  and\n"
        "  set BEDTOOLS_BIN in Section 1 to the full path.")
BEDTOOLS_VERSION = subprocess.run([BEDTOOLS, "--version"], capture_output=True,
                                  text=True).stdout.strip()
print(f"using {BEDTOOLS}  ({BEDTOOLS_VERSION})")

# Every reproducible cluster is written to a single query BED (one bedtools call per panel,
# not one call per cluster).
rep = clusters.loc[clusters.reproducible].copy()
rep["cluster_id"] = rep.index.astype(str)
query_bed = OUTDIR / "_tmp_reproducible_clusters_query.bed"
with open(query_bed, "w") as f:
    for cid, s, e in zip(rep["cluster_id"], rep["start"], rep["end"]):
        f.write(f"{CAGE_CHROM}\t{s}\t{e}\t{cid}\n")

def overlap_hit_ids(query_bed, bed_path, bedtools_bin):
    if not Path(bed_path).exists():
        raise FileNotFoundError(bed_path)
    out = subprocess.run([bedtools_bin, "intersect", "-a", str(query_bed), "-b", str(bed_path), "-wa", "-u"],
                          capture_output=True, text=True, check=True)
    return {line.split("\t")[3] for line in out.stdout.strip().split("\n") if line}

for name, bed_path in CHROMATIN_PANELS.items():
    hits = overlap_hit_ids(query_bed, bed_path, BEDTOOLS)
    clusters[name] = [
        (cid in hits) if rep_flag else None
        for rep_flag, cid in zip(clusters["reproducible"], clusters.index.astype(str))
    ]

# Registry V4 classification (liver.all) - report the class, not just presence/absence,
# since "Low-DNase" vs "PLS" is itself informative (see Part B, Section 9).
if not LIVER_CCRE_BED.exists():
    raise FileNotFoundError(f"Missing Registry V4 liver.all file: {LIVER_CCRE_BED}")
out = subprocess.run([BEDTOOLS, "intersect", "-a", str(query_bed), "-b", str(LIVER_CCRE_BED), "-wa", "-wb"],
                      capture_output=True, text=True, check=True)
registry_classes = {}
for line in out.stdout.strip().split("\n"):
    if not line:
        continue
    fields = line.split("\t")
    cid = fields[3]
    cls = fields[-2]   # ENCODE cCRE BED: class is 2nd-to-last column
    registry_classes.setdefault(cid, set()).add(cls)

clusters["registry_liver_all_class"] = [
    ", ".join(sorted(registry_classes[cid])) if (rep_flag and cid in registry_classes)
    else (None if not rep_flag else "none")
    for rep_flag, cid in zip(clusters["reproducible"], clusters.index.astype(str))
]
query_bed.unlink()

clusters.loc[clusters.reproducible].sort_values("tpm_max", ascending=False)[
    ["summit", "tpm_max", "nearest_transcript_category"] + list(CHROMATIN_PANELS) + ["registry_liver_all_class"]
]


### A7. Output table

Every consensus cluster found in the window is written out, not only the ones discussed
above - this is the unbiased-scan deliverable. Non-reproducible clusters are included with
chromatin/annotation columns left blank (that annotation step is run only on reproducible
clusters, to keep runtime and the bedtools call count reasonable; the raw TPM values for
every cluster, reproducible or not, are always reported).

In [ ]:
out_cols = (["strand", "start", "end", "width_bp", "summit"]
            + [f"tpm_{l}" for l in CAGE_FILES] + ["tpm_max", "n_hep_donors_detected", "reproducible"]
            + ["nearest_transcript_id", "nearest_transcript_name",
               "nearest_transcript_category", "dist_to_nearest_tx_tss_bp"]
            + list(CHROMATIN_PANELS) + ["registry_liver_all_class"])
clusters_out = clusters[out_cols].sort_values("tpm_max", ascending=False)
clusters_out.to_csv(OUTDIR / "SLC38A4_CAGE_TSS_scan.csv", index=False)
print(f"written: SLC38A4_CAGE_TSS_scan.csv ({len(clusters_out)} clusters, "
      f"{clusters_out['reproducible'].sum()} reproducible)")

tx.to_csv(OUTDIR / "SLC38A4_CAGE_transcript_models.csv", index=False)
print("written: SLC38A4_CAGE_transcript_models.csv")


### A8. Locus plot

House style, as in the other figure notebooks for this paper. Top panel: per-sample CAGE
signal (TPM, symlog scale - the dynamic range across clusters spans more than two orders of
magnitude, so a linear axis would make every cluster but the largest invisible). Bottom
panel: GENCODE transcript models, coloured by category, with rs4994910 marked for context
only.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 10, 'axes.linewidth': 1.0,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'axes.unicode_minus': False,
})
AXTITLE = 11
CAT_COLORS = {
    "canonical (MANE Select)": "#D73027",
    "canonical-adjacent": "#FC8D59",
    "extended (5' of canonical)": "#4575B4",
    "internal/shorter than canonical": "#91BFDB",
    "NMD": "#999999",
    "lncRNA": "#B39DDB",
}
SAMPLE_COLORS = {
    "Hepatocyte_donor1": "#1B9E77", "Hepatocyte_donor2": "#D95F02",
    "Hepatocyte_donor3": "#7570B3", "Liver_adult_pool1": "#666666",
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1.4], "hspace": 0.08})

for label in CAGE_FILES:
    sub = clusters[clusters[f"tpm_{label}"] > 0]
    ax1.vlines(sub["summit"], 0, sub[f"tpm_{label}"], color=SAMPLE_COLORS[label],
               linewidth=1.1, alpha=0.85, label=label.replace("_", " "))
ax1.set_yscale("symlog", linthresh=1)
ax1.set_ylabel("CAGE signal (TPM)", fontsize=AXTITLE)
ax1.axvline(RS4994910_POS, color="black", linestyle=":", linewidth=0.9, zorder=1)
ax1.text(RS4994910_POS, ax1.get_ylim()[1], " rs4994910", fontsize=8, va="top", ha="left")
ax1.legend(fontsize=7.5, frameon=False, ncol=2, loc="upper left")

plotted_ids = set()
y = 0
for _, row in tx.sort_values("tss").iterrows():
    if row.transcript_id in plotted_ids:
        continue
    plotted_ids.add(row.transcript_id)
    color = CAT_COLORS.get(row.category, "#333333")
    ax2.plot([row.start, row.end], [y, y], color=color, linewidth=1.2, solid_capstyle="butt")
    marker = ">" if row.strand == 1 else "<"
    ax2.plot(row.tss, y, marker=marker, color=color, markersize=3)
    y += 1
ax2.set_ylim(-1, y)
ax2.set_yticks([])
ax2.set_xlabel(f"{CAGE_CHROM} position (GRCh38)", fontsize=AXTITLE)
ax2.axvline(RS4994910_POS, color="black", linestyle=":", linewidth=0.9, zorder=1)

handles = [plt.Line2D([0], [0], color=c, lw=2, label=k) for k, c in CAT_COLORS.items()
           if k in tx["category"].values]
ax2.legend(handles=handles, fontsize=7, frameon=False, ncol=3, loc="upper left",
           bbox_to_anchor=(0, -0.35))

ax1.set_xlim(*CAGE_WINDOW)
plt.tight_layout()
fig.savefig(OUTDIR / "SuppFig_SLC38A4_CAGE_locus_scan.pdf", bbox_inches="tight")
plt.show()
print("written: SuppFig_SLC38A4_CAGE_locus_scan.pdf")


### A9. Provenance (Part A)

In [ ]:
provenance_cage = {
    "notebook": "Fig5_LD_expansion_cCRE_filtering.ipynb (Part A -- consolidated from the "
                "former SuppFig_SLC38A4_CAGE_TSS_scan.ipynb)",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "cage_source": "FANTOM5 (CNhs accessions in filenames), hg38 CTSS",
    "window": {"chrom": CAGE_CHROM, "start": CAGE_WINDOW[0], "end": CAGE_WINDOW[1]},
    "cluster_gap_bp": CLUSTER_GAP_BP,
    "min_cluster_tpm": MIN_CLUSTER_TPM,
    "min_hep_donors_for_reproducible": MIN_HEP_DONORS,
    "library_sizes": libsize,
    "gene_id": GENE_ID,
    "canonical_transcript_id": CANONICAL_TRANSCRIPT_ID,
    "canonical_tss_grch38": canon_tss,
    "n_consensus_clusters": int(len(clusters)),
    "n_reproducible_clusters": int(clusters["reproducible"].sum()),
    "chromatin_panels": {k: str(v) for k, v in CHROMATIN_PANELS.items()},
    "registry_bed": str(LIVER_CCRE_BED),
}
with open(OUTDIR / "SLC38A4_CAGE_TSS_scan_provenance.json", "w") as f:
    json.dump(provenance_cage, f, indent=2)
print("written: SLC38A4_CAGE_TSS_scan_provenance.json")


### A10. Notes (Part A)

- **This is a descriptive scan, not a statistical test.** With one pooled liver library and
  three hepatocyte donors there is no meaningful way to assign a p-value to "is this cluster
  real"; reproducibility across donors at a fixed TPM threshold is the criterion, and it is a
  threshold choice, not a hypothesis test. Report clusters, not significance.
- **A cluster with strong CAGE support and no chromatin support (or vice versa) is a genuine
  finding, not a contradiction to explain away.** CAGE, ATAC and H3K27ac each fail differently
  - a true positive by one assay and a miss by another is expected some of the time; check the
  chromatin columns per cluster rather than requiring all three to agree before trusting any
  of them.
- **The Registry V4 class is reported per cluster deliberately, including "Low-DNase" where it
  occurs** - the same discordance already documented for the rs4994910 element in Part B below
  (Registry DNase-based calls vs directly-measured ATAC).
- **If a cluster of interest turns out to be inside the rs4994910 element itself**, that bears
  directly on how that element has been described in the manuscript text so far.
- Any RNA-seq with exon- or junction-level resolution would be the next, more direct line of
  evidence for isoform usage (CAGE only marks capped 5' ends).


## Table S13 — CAGE data evidence of expression for SLC38A4 transcript isoforms

In [ ]:
# ---- Table S13 -- CAGE TSS clusters, ranked by signal ----
st13 = clusters_out[clusters_out["reproducible"]].copy()

_RENAME_S13 = {
    "strand": "strand", "start": "start_grch38", "end": "end_grch38", "summit": "summit_grch38",
    "width_bp": "width_bp",
    "tpm_Hepatocyte_donor1": "tpm_hepatocyte_donor1", "tpm_Hepatocyte_donor2": "tpm_hepatocyte_donor2",
    "tpm_Hepatocyte_donor3": "tpm_hepatocyte_donor3", "tpm_Liver_adult_pool1": "tpm_liver_adult_pool1",
    "tpm_max": "tpm_max", "n_hep_donors_detected": "n_hepatocyte_donors_detected_ge0.5TPM",
    "nearest_transcript_id": "nearest_transcript_id", "nearest_transcript_name": "nearest_transcript_name",
    "nearest_transcript_category": "nearest_transcript_category",
    "dist_to_nearest_tx_tss_bp": "dist_to_nearest_transcript_tss_bp",
    "PHH_ATAC_replicated": "overlaps_PHH_ATAC_replicated_peak",
    "ENCODE_liver_ATAC_replicated": "overlaps_ENCODE_liver_ATAC_replicated_peak",
    "H3K27ac_replicated_in2": "overlaps_H3K27ac_replicated_peak_in2",
    "registry_liver_all_class": "registry_liver_cCRE_class",
}
st13 = st13.rename(columns=_RENAME_S13)
st13["nearest_transcript_category"] = st13["nearest_transcript_category"].fillna("unannotated")
st13["nearest_transcript_id"] = st13["nearest_transcript_id"].fillna("unannotated")
st13["nearest_transcript_name"] = st13["nearest_transcript_name"].fillna("unannotated")

st13 = st13.sort_values("tpm_max", ascending=False).reset_index(drop=True)
st13.insert(0, "rank", range(1, len(st13) + 1))
_col_order = ["rank"] + [c for c in st13.columns if c != "rank"]
st13 = st13[_col_order]

n_total_clusters = int(len(clusters_out))
n_reproducible = int(st13.shape[0])

ST13_PATH = "Table_S13_CAGE_TSS_clusters.xlsx"
_readme_s13 = pd.DataFrame({"Table S13 - CAGE data evidence of expression for "
                            "SLC38A4 transcript isoforms in liver and hepatocytes": [
    "Reproducible FANTOM5 CAGE transcription start site (TSS) clusters at the SLC38A4 locus, "
    "ranked by maximum signal.",
    f"FANTOM5 CAGE data (3 primary human hepatocyte donors + 1 pooled adult liver track; "
    f"Supplementary Methods 5.1) were scanned across {CAGE_CHROM}:{CAGE_WINDOW[0]:,}-"
    f"{CAGE_WINDOW[1]:,} (GRCh38). CAGE tag clusters were called per sample and strand "
    f"(maximum intra-cluster gap {CLUSTER_GAP_BP} bp); {n_total_clusters} consensus clusters "
    f"were identified across the locus. A cluster was retained as reproducible if detected at "
    f">={MIN_CLUSTER_TPM} TPM in at least {MIN_HEP_DONORS} of the 3 hepatocyte donors: "
    f"{n_reproducible} of {n_total_clusters} clusters met this threshold and are listed here, "
    "one row per cluster.",
    "Ranked by tpm_max (the highest TPM value across the four CAGE samples) in descending "
    "order. TPM values are rounded to 2 decimal places; source (unrounded) values are in "
    "SLC38A4_CAGE_TSS_scan.csv.",
    "Rows 1-3 are the three clusters discussed in Supplementary Results Section 6 and Methods "
    "5.1: rank 1 is the highest-signal cluster, associated exclusively with a "
    "nonsense-mediated-decay (NMD) transcript biotype and excluded from Fig. 5A/Fig. S12A; "
    "ranks 2-3 are the canonical and extended-5' transcription start sites shown in "
    "Fig. 5A/Fig. S12A.",
    "Chromatin overlap columns (overlaps_PHH_ATAC_replicated_peak, "
    "overlaps_ENCODE_liver_ATAC_replicated_peak, overlaps_H3K27ac_replicated_peak_in2) indicate "
    "overlap with a replicated peak from the corresponding dataset (Table S1). "
    "registry_liver_cCRE_class is this position's classification in the ENCODE Registry of "
    "candidate cis-regulatory elements (PLS = promoter-like signature; dELS = distal "
    "enhancer-like signature; Low-DNase = fails the DNase threshold; none = no registry entry "
    "at this position).",
    "nearest_transcript_* columns give the closest GENCODE/Ensembl transcript TSS to the "
    "cluster summit; clusters with no transcript TSS nearby are marked \"unannotated\". "
    "Several GENCODE transcripts share an essentially identical annotated TSS (the "
    "\"extended (5' of canonical)\" family); where a cluster's nearest transcript is tied on "
    "distance between two or more such transcripts, the specific transcript ID reported can "
    "vary between runs (depending on the order Ensembl returns them in), but the reported "
    "category and distance do not.",
    "Source data: SLC38A4_CAGE_TSS_scan.csv, generated by this notebook (Part A, Sections A1-A9; "
    "consolidated from the former SuppFig_SLC38A4_CAGE_TSS_scan.ipynb).",
]})
with pd.ExcelWriter(ST13_PATH, engine="openpyxl") as xw:
    _readme_s13.to_excel(xw, sheet_name="README", index=False)
    st13.to_excel(xw, sheet_name="CAGE_TSS_clusters", index=False)

print(f"{n_reproducible} reproducible clusters (of {n_total_clusters} total). Saved: {ST13_PATH}")
st13.head(5)


## Part B — Lead-variant LD expansion and cCRE-based variant prioritisation (Supplementary Methods 5.3)

## 2. Locus definition — resolve the lead variant on GRCh38

In [ ]:
def ensembl_variant(rsid):
    """GRCh38 primary-assembly mapping for an rsID, from Ensembl (dbSNP-backed)."""
    rec = get_json(f"{ENSEMBL}/variation/human/{rsid}",
                   params={"content-type": "application/json"},
                   headers={"Accept": "application/json"})
    PRIMARY = {str(c) for c in range(1, 23)} | {"X", "Y", "MT"}
    maps = [m for m in rec.get("mappings", [])
            if str(m.get("assembly_name", "")).startswith("GRCh38")
            and str(m.get("seq_region_name")) in PRIMARY]
    if not maps:
        raise ApiError(f"{rsid}: no primary GRCh38 mapping")
    if len({(m["seq_region_name"], m["start"]) for m in maps}) > 1:
        print(f"  WARNING: {rsid} has multiple primary GRCh38 mappings - inspect manually")
    m = maps[0]
    return dict(rsid=rec.get("name", rsid), chrom=str(m["seq_region_name"]),
                pos=int(m["start"]), alleles=m.get("allele_string"),
                strand=m.get("strand"))

lead = ensembl_variant(LEAD_RSID)
print(f"  {lead['rsid']:12s} chr{lead['chrom']}:{lead['pos']:,}  "
      f"alleles={lead['alleles']}  strand={lead['strand']}")

CHROM  = "chr" + lead["chrom"]
WIN_LO = max(0, lead["pos"] - WINDOW_KB * 1000)
WIN_HI = lead["pos"] + WINDOW_KB * 1000
print(f"\nlocus window: {CHROM}:{WIN_LO:,}-{WIN_HI:,}  (+/-{WINDOW_KB} kb, GRCh38)")


## 3. LD expansion — Source A: LDlink `LDproxy` (GBR)

LDproxy returns a TSV. Note that LDlink signals errors in the response body with HTTP 200,
so the body is checked before parsing.


In [ ]:
import io as _io

def ldlink_proxies():
    if not LDLINK_TOKEN:
        print("skipped - no LDLINK_TOKEN")
        return frame([], LD_COLS, "")
    txt = get_text(f"{LDLINK}/ldproxy", params={
        "var": LEAD_RSID, "pop": POP, "r2_d": "r2",
        "window": WINDOW_KB * 2 * 1000,          # LDproxy window is the full span
        "genome_build": "grch38", "token": LDLINK_TOKEN})
    if txt.strip().startswith("{") or "error" in txt[:200].lower():
        raise ApiError("LDlink returned an error:\n    " + txt[:300])
    raw = pd.read_csv(_io.StringIO(txt), sep="\t")
    print(f"  LDproxy returned {len(raw)} rows; columns: {list(raw.columns)}")

    def coord(c):
        c = str(c)
        return c.split(":") if ":" in c else (None, None)
    recs = []
    for r in raw.itertuples():
        ch, po = coord(getattr(r, "Coord", ""))
        recs.append(dict(rsid=getattr(r, "RS_Number", None),
                         chrom=(ch or "").replace("chr", "") or None,
                         pos_grch38=po,
                         alleles=getattr(r, "Alleles", None),
                         maf=getattr(r, "MAF", None),
                         distance_bp=getattr(r, "Distance", None),
                         r2_ldlink=getattr(r, "R2", None),
                         dprime_ldlink=getattr(r, "Dprime", None)))
    df = frame(recs, LD_COLS, "ldlink")
    df = df[df["rsid"].astype(str).str.startswith("rs")]     # drop '.' placeholders
    return df.reset_index(drop=True)

ld_a = ldlink_proxies()
print(f"\nLDlink proxies: {len(ld_a)}")
if len(ld_a):
    print(ld_a.sort_values("r2_ldlink", ascending=False)
              .head(12)[["rsid","pos_grch38","alleles","maf","distance_bp","r2_ldlink"]]
              .to_string(index=False))


## 4. LD expansion — Source B: Ensembl REST

`1000GENOMES:phase_3:GBR` — the same reference population, computed independently.

This is not only a cross-check. Each service returns proxies the other does not, so the union
covers more of the locus than either alone; and where both report a variant, the two R²
estimates give a direct measure of how robust its threshold membership is. Ensembl also needs
no credentials, so this source works before an LDlink token is in place.


In [ ]:
def ensembl_proxies():
    js = get_json(f"{ENSEMBL}/ld/human/{LEAD_RSID}/1000GENOMES:phase_3:{POP}",
                  params={"r2": 0.1, "window_size": min(WINDOW_KB * 2, 500),
                          "content-type": "application/json"},
                  headers={"Accept": "application/json"})
    recs = []
    for e in js:
        other = e["variation2"] if e.get("variation1") == LEAD_RSID else e.get("variation1")
        recs.append(dict(rsid=other, r2_ensembl=e.get("r2"), dprime_ensembl=e.get("d_prime")))
    df = frame(recs, LD_COLS, "ensembl")
    return (df.dropna(subset=["rsid"])
              .sort_values("r2_ensembl", ascending=False)
              .drop_duplicates("rsid")
              .reset_index(drop=True))

ld_b = ensembl_proxies()
print(f"Ensembl proxies: {len(ld_b)}")
print(ld_b.sort_values("r2_ensembl", ascending=False)
          .head(12)[["rsid","r2_ensembl","dprime_ensembl"]].to_string(index=False))


## 5. Reconcile the two LD sources

Proxies are unioned, not intersected.
Where both report R², the difference is shown.


In [ ]:
# ---------------------------------------------------------------------------
# Reconcile the two LD sources at SNP level: chromosome, position AND alleles.
#
# Keying on rsID double-counts variants, because the services often return
# different (co-located or merged) identifiers for the same SNP. Keying on
# position alone is also wrong: distinct variants can share a start coordinate
# (multi-allelic sites, or an SNV and an indel), and merging them would combine
# different SNPs into one row.
#
# Alleles are matched allowing for the fact that Ensembl commonly reports the
# union of all observed alleles at a site (e.g. A/G/T) where LDlink reports the
# pair actually assayed (A/G). A strict set equality would therefore fail to
# match the same SNP. Matching proceeds in three stages, and each row records
# how it was matched.
# ---------------------------------------------------------------------------

def allele_key(s):
    """'(T/C)' or 'T/C/A' -> frozenset({'T','C','A'}); unknown -> None"""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    a = {x.strip().upper() for x in str(s).strip("() ").split("/") if x.strip()}
    return frozenset(a) or None

def compat(x, y):
    """How two allele sets relate. -> ('exact'|'subset'|'partial'|'unverified'|'incompatible')"""
    if x is None or y is None:            return "unverified"
    if x == y:                            return "exact"
    if x <= y or y <= x:                  return "subset"
    if len(x & y) >= 2:                   return "partial"
    return "incompatible"

# ---- resolve GRCh38 coordinates and alleles for the Ensembl proxies (batched) ----
need = sorted(set(ld_b["rsid"].dropna()))
print(f"resolving GRCh38 coordinates for {len(need)} Ensembl proxy rsID(s) in batches...")
coord = {}
PRIMARY_CHR = {str(c) for c in range(1, 23)} | {"X", "Y", "MT"}
for i in range(0, len(need), 200):
    recs = post_json(f"{ENSEMBL}/variation/homo_sapiens",
                      headers={"Content-Type": "application/json",
                               "Accept": "application/json"},
                      json={"ids": need[i:i+200], "pops": 0, "genotypes": 0,
                            "phenotypes": 0}, timeout=120)
    for rs, rec in (recs or {}).items():
        if not rec: continue
        maps = [m for m in rec.get("mappings", [])
                if str(m.get("assembly_name", "")).startswith("GRCh38")
                and str(m.get("seq_region_name")) in PRIMARY_CHR]
        if maps:
            coord[rs] = (str(maps[0]["seq_region_name"]), int(maps[0]["start"]),
                         maps[0].get("allele_string"))
    time.sleep(0.2)
print(f"  resolved {len(coord)}/{len(need)}")

b = ld_b.copy()
b["chrom"]      = [coord.get(r, (None,None,None))[0] for r in b["rsid"]]
b["pos_grch38"] = [coord.get(r, (None,None,None))[1] for r in b["rsid"]]
b["alleles"]    = [coord.get(r, (None,None,None))[2] for r in b["rsid"]]
drop = int(b["pos_grch38"].isna().sum())
if drop: print(f"  {drop} Ensembl proxy/proxies unplaced and dropped")
b = b.dropna(subset=["pos_grch38"])

A = ld_a.dropna(subset=["pos_grch38"]).copy()
for f in (A, b):
    f["chrom"] = f["chrom"].astype(str)
    f["pos_grch38"] = pd.to_numeric(f["pos_grch38"], errors="coerce").astype("Int64")
    f["akey"] = f["alleles"].map(allele_key)

A = (A.sort_values("r2_ldlink", ascending=False)
       .drop_duplicates(["chrom","pos_grch38","akey"])
       [["chrom","pos_grch38","akey","rsid","alleles","maf","r2_ldlink","dprime_ldlink"]]
       .rename(columns={"rsid":"rsid_ldlink","alleles":"alleles_ldlink"}))
B = (b.sort_values("r2_ensembl", ascending=False)
       .drop_duplicates(["chrom","pos_grch38","akey"])
       [["chrom","pos_grch38","akey","rsid","alleles","r2_ensembl","dprime_ensembl"]]
       .rename(columns={"rsid":"rsid_ensembl","alleles":"alleles_ensembl",
                        "akey":"akey_ens"}))

# ---- stage 1: exact SNP match (chrom, pos, identical allele set) ----
s1 = A.merge(B, left_on=["chrom","pos_grch38","akey"],
             right_on=["chrom","pos_grch38","akey_ens"], how="inner")
s1["allele_match"] = "exact"

# ---- stage 2: same position, allele sets compatible (subset / partial) ----
restA = A[~A.set_index(["chrom","pos_grch38"]).index.isin(
            s1.set_index(["chrom","pos_grch38"]).index)]
restB = B[~B.set_index(["chrom","pos_grch38"]).index.isin(
            s1.set_index(["chrom","pos_grch38"]).index)]
cand = restA.merge(restB, on=["chrom","pos_grch38"], how="inner")
cand["allele_match"] = [compat(x, y) for x, y in zip(cand["akey"], cand["akey_ens"])]
s2 = cand[cand["allele_match"].isin(["subset","partial","unverified"])].copy()
# one pairing per position, preferring the strongest evidence of identity
rank = {"subset":0, "partial":1, "unverified":2}
s2 = (s2.assign(_r=s2["allele_match"].map(rank))
        .sort_values("_r").drop_duplicates(["chrom","pos_grch38"]).drop(columns="_r"))
incompat = cand[cand["allele_match"] == "incompatible"]

# ---- stage 3: anything still unmatched stays as its own SNP ----
matched = pd.concat([s1, s2], ignore_index=True)
keyA = set(zip(matched["chrom"], matched["pos_grch38"], matched["akey"]))
keyB = set(zip(matched["chrom"], matched["pos_grch38"], matched["akey_ens"]))
onlyA = A[[k not in keyA for k in zip(A["chrom"], A["pos_grch38"], A["akey"])]].copy()
onlyB = B[[k not in keyB for k in zip(B["chrom"], B["pos_grch38"], B["akey_ens"])]].copy()
onlyA["allele_match"] = "ldlink only"
onlyB["allele_match"] = "ensembl only"

ld = pd.concat([matched, onlyA, onlyB], ignore_index=True, sort=False)

ld["rsid"]     = ld["rsid_ldlink"].fillna(ld["rsid_ensembl"])
ld["rsid_alt"] = [e if (pd.notna(l) and pd.notna(e) and l != e) else None
                  for l, e in zip(ld["rsid_ldlink"], ld["rsid_ensembl"])]
ld["alleles"]  = ld["alleles_ldlink"].fillna(ld["alleles_ensembl"])
ld["distance_bp"] = ld["pos_grch38"] - lead["pos"]
ld["r2_max"]   = ld[["r2_ldlink","r2_ensembl"]].max(axis=1)
ld["r2_delta"] = (ld["r2_ldlink"] - ld["r2_ensembl"]).abs()
ld = ld[ld["pos_grch38"].between(WIN_LO, WIN_HI)].copy()
ld["passes_threshold"] = ld["r2_max"] >= R2_THRESHOLD
ld["threshold_is_source_dependent"] = (
    ((ld["r2_ldlink"]  >= R2_THRESHOLD) & (ld["r2_ensembl"] < R2_THRESHOLD)) |
    ((ld["r2_ensembl"] >= R2_THRESHOLD) & (ld["r2_ldlink"]  < R2_THRESHOLD)))

assert not ld.duplicated(["chrom","pos_grch38","alleles"]).any(), "duplicate SNPs remain"

# ---------------------------- reconciliation summary ----------------------------
print(f"\n{len(ld)} distinct SNP(s) in window, matched as follows:")
for k, n in ld["allele_match"].value_counts().items():
    print(f"  {k:<14} {n:>5}")
if len(incompat):
    print(f"\n  {len(incompat)} position(s) where the two services reported "
          "INCOMPATIBLE alleles;")
    print("  these are treated as distinct SNPs rather than merged:")
    print(incompat[["chrom","pos_grch38","rsid_ldlink","alleles_ldlink",
                    "rsid_ensembl","alleles_ensembl"]].head(10).to_string(index=False))
n_alt = int(ld["rsid_alt"].notna().sum())
print(f"\n  {n_alt} matched SNP(s) carry different rsIDs between services (kept in rsid_alt)")
both = ld["r2_ldlink"].notna() & ld["r2_ensembl"].notna()
if both.sum() > 1:
    sub = ld[both]
    print(f"  agreement where both report: r = {sub['r2_ldlink'].corr(sub['r2_ensembl']):.5f}, "
          f"mean |diff| = {sub['r2_delta'].mean():.4f}, max = {sub['r2_delta'].max():.4f}")
print(f"\n{int(ld['passes_threshold'].sum())} SNP(s) with R2 >= {R2_THRESHOLD}")
amb = ld[ld["threshold_is_source_dependent"]]
print(f"{len(amb)} whose inclusion at the threshold depends on the service used"
      + (":" if len(amb) else ""))
if len(amb):
    print(amb[["rsid","pos_grch38","alleles","r2_ldlink","r2_ensembl"]].to_string(index=False))

disc = ld[ld["r2_delta"] > 0.1].dropna(subset=["r2_delta"])
if len(disc):
    print(f"\n{len(disc)} SNP(s) where the services differ by R2 > 0.1:")
    print(disc[["rsid","rsid_alt","pos_grch38","alleles","r2_ldlink","r2_ensembl",
                "r2_delta"]].to_string(index=False))

ld.sort_values("r2_max", ascending=False).head(20)[
    ["rsid","rsid_alt","pos_grch38","alleles","allele_match",
     "r2_ldlink","r2_ensembl","passes_threshold"]]


## 6. cCRE Registry V4 — discover the track and record its version

Rather than hard-coding a track name that may change, the available hg38 tracks are
searched for cCRE candidates and their `longLabel` printed. **Read the labels and confirm
V4** before continuing — the label is written to the provenance file, so the version used
is recorded rather than assumed. This is the step that was missing from Supplementary
Methods 5.3.


In [ ]:
tracks = get_json(f"{UCSC}/list/tracks", params={"genome": GENOME})[GENOME]

def walk(d, depth=0):
    for name, meta in d.items():
        if not isinstance(meta, dict):
            continue
        yield name, meta
        for k, v in meta.items():
            if isinstance(v, dict) and "shortLabel" in v:
                yield from walk({k: v}, depth + 1)

hits = []
for name, meta in walk(tracks):
    blob = " ".join(str(meta.get(k, "")) for k in ("shortLabel", "longLabel"))
    if any(t in blob.lower() or t in name.lower() for t in ("ccre", "cis-regulatory")):
        hits.append((name, meta.get("shortLabel", ""), meta.get("longLabel", "")))

print(f"{len(hits)} candidate cCRE track(s) on {GENOME}:\n")
for name, short, long_ in sorted(set(hits)):
    print(f"  {name:28s} | {short[:34]:34s} | {long_[:80]}")

print("\n--> set CCRE_TRACK below to the V4 track, then run the next cell.")
CCRE_TRACK = "cCREregistry"   # e.g. "encodeCcreCombined" - CONFIRM the longLabel says V4
print(CCRE_TRACK)

## 7. Fetch cCREs across the locus

In [ ]:
assert CCRE_TRACK, "Set CCRE_TRACK from the discovery output above before running this cell."

CCRE_LABEL = next((l for n, s, l in hits if n == CCRE_TRACK), "(label not found)")
print(f"track      : {CCRE_TRACK}")
print(f"longLabel  : {CCRE_LABEL}")
if "v4" not in CCRE_LABEL.lower():
    print("\n*** WARNING: the track label does not mention V4. Confirm the registry\n"
          "    version before using these results. ***")

resp = get_json(f"{UCSC}/getData/track", params={
    "genome": GENOME, "track": CCRE_TRACK,
    "chrom": CHROM, "start": WIN_LO, "end": WIN_HI})

payload = resp.get(CCRE_TRACK, resp.get("data", []))
if isinstance(payload, dict):            # composite track -> subtrack dict
    print("\ncomposite track; subtracks:", list(payload)[:10])
    payload = [row for sub in payload.values() if isinstance(sub, list) for row in sub]

raw = pd.DataFrame(payload)
print(f"\n{len(raw)} cCRE record(s) in window; fields: {list(raw.columns)}")

def pick(df, *names):
    for n in names:
        if n in df.columns: return df[n]
    return pd.Series([None] * len(df), index=df.index)

ccre = frame([{
    "ccre_accession": a, "chrom": c, "start": s, "end": e,
    "ccre_class": cl, "ccre_label": lb,
} for a, c, s, e, cl, lb in zip(
    pick(raw, "name", "accession", "itemName"),
    pick(raw, "chrom"), pick(raw, "chromStart", "start"), pick(raw, "chromEnd", "end"),
    pick(raw, "ucscLabel", "encodeLabel", "cCRE_class", "ccre_class",
              "class", "group", "itemName"),
    pick(raw, "description", "ucscLabel", "name"),
)], CCRE_COLS, "ccre")

if ccre["ccre_class"].isna().all():
    print("\n*** WARNING: no class field recognised in this track's response. ***")
    print(f"    Fields returned: {list(raw.columns)}")
    print("    ccre_class is empty, so the Registry-wide class is NOT available from UCSC\n"
          "    here. Add the correct field name to the pick() call above, or use the\n"
          "    liver-aggregate class added below - do not report a class you cannot source.")
else:
    print("\ncCRE classes present in the window:")
    print(ccre["ccre_class"].value_counts(dropna=False).to_string())

# --- independent, clearly-labelled class from the liver.all aggregate ---
# Kept in its own column rather than back-filling ccre_class, so the provenance of every
# reported class is unambiguous.
_agg = LIVER_AGGREGATE.get("liver.all")
if _agg and Path(_agg).exists():
    lut = {}
    with open(_agg) as fh:
        for line in fh:
            f = line.rstrip("\n").split("\t")
            if len(f) >= 10:
                lut[f[3]] = f[9]
    ccre["ccre_class_liver_all"] = ccre["ccre_accession"].map(lut)
    n = int(ccre["ccre_class_liver_all"].notna().sum())
    print(f"\nliver.all aggregate class attached for {n}/{len(ccre)} element(s)")
else:
    ccre["ccre_class_liver_all"] = None
    print("\nliver.all aggregate not found - ccre_class_liver_all left empty")

ccre.head(10)


## 8. Intersect proxies with cCREs

Overlaps are computed with **`bedtools intersect`**, which is required — the notebook stops if
it is not found, rather than substituting a different method and reporting it as though
nothing had changed.

The same overlap is also computed in pure Python and the two are asserted to agree. That is
not a fallback: it is a check that the BED files were written correctly, since the commonest
error here is a coordinate-convention mistake (cCRE intervals are 0-based half-open, variant
positions are 1-based) which would produce a clean but silently shifted result.

Install with `conda install -c conda-forge -c bioconda bedtools`, or `brew install bedtools`
on Apple Silicon where bioconda has no `osx-arm64` build. If the binary is installed but not
on the kernel's `PATH`, set `BEDTOOLS_BIN` in section 1.


In [ ]:
def overlap_pandas(variants, elements):
    """Half-open [start, end) overlap. cCRE BED coordinates are 0-based; variant
    positions are 1-based, so a variant at 1-based p occupies 0-based [p-1, p)."""
    out = []
    el = elements.dropna(subset=["start", "end"])
    for v in variants.itertuples():
        if pd.isna(v.pos_grch38):
            continue
        p0 = int(v.pos_grch38) - 1
        hit = el[(el["start"] <= p0) & (p0 < el["end"])]
        if len(hit) == 0:
            out.append({"rsid": v.rsid, "ccre_accession": None, "ccre_class": None,
                        "ccre_class_liver_all": None,
                        "ccre_start": pd.NA, "ccre_end": pd.NA})
        else:
            for h in hit.itertuples():
                out.append({"rsid": v.rsid, "ccre_accession": h.ccre_accession,
                            "ccre_class": h.ccre_class,
                            "ccre_class_liver_all": getattr(h, "ccre_class_liver_all", None),
                            "ccre_start": h.start, "ccre_end": h.end})
    return pd.DataFrame(out)

# independent calculation, used below to verify the bedtools result
ov = overlap_pandas(ld, ccre)
n_in = ov["ccre_accession"].notna().sum()
print(f"interval calculation: {n_in} proxy-cCRE overlap(s) "
      f"across {ov['rsid'].nunique()} proxies")

# --- bedtools is the method of record; the calculation above is the correctness check ---
import shutil, subprocess, tempfile
BEDTOOLS = BEDTOOLS_BIN or shutil.which("bedtools")
if not BEDTOOLS:
    raise RuntimeError(
        "bedtools not found.\n"
        "  Install:  conda install -c conda-forge -c bioconda bedtools\n"
        "            (on Apple Silicon, where bioconda has no osx-arm64 build:\n"
        "             brew install bedtools)\n"
        "  If installed but not on this kernel's PATH - common when Jupyter is started\n"
        "  from an app rather than a terminal - check with  !bedtools --version  and set\n"
        "  BEDTOOLS_BIN in section 1 to the full path.")
BEDTOOLS_VERSION = subprocess.run([BEDTOOLS, "--version"], capture_output=True,
                                  text=True).stdout.strip()
BEDTOOLS_USED = True
print(f"using {BEDTOOLS}  ({BEDTOOLS_VERSION})")
if True:
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        vb = ld.dropna(subset=["pos_grch38"]).copy()
        (td / "v.bed").write_text("".join(
            f"{CHROM}\t{int(p)-1}\t{int(p)}\t{r}\n"
            for r, p in zip(vb["rsid"], vb["pos_grch38"])))
        cb = ccre.dropna(subset=["start", "end"])
        (td / "c.bed").write_text("".join(
            f"{c}\t{int(s)}\t{int(e)}\t{a}\n"
            for c, s, e, a in zip(cb["chrom"], cb["start"], cb["end"], cb["ccre_accession"])))
        res = subprocess.run([BEDTOOLS, "intersect", "-a", str(td/"v.bed"),
                              "-b", str(td/"c.bed"), "-wa", "-wb"],
                             capture_output=True, text=True)
        bt = {(l.split("\t")[3], l.split("\t")[7]) for l in res.stdout.strip().split("\n") if l}
    pn = {(r.rsid, r.ccre_accession) for r in ov.itertuples() if pd.notna(r.ccre_accession)}
    print(f"bedtools overlap    : {len(bt)} pair(s)")
    print("cross-check agrees  :", "YES" if bt == pn else f"NO - differences: {bt ^ pn}")
    assert bt == pn, (
        "bedtools and the independent interval calculation disagree. This almost always "
        "means a coordinate-convention error in the BED files written above (cCRE "
        "intervals are 0-based half-open; variant positions are 1-based) - do not proceed.")

# ---- build the per-proxy x cCRE annotation table used by the next sections ----
ann = ld.merge(ov, on="rsid", how="left")
ann["in_ccre"] = ann["ccre_accession"].notna()
print(f"\nann: {len(ann)} proxy-cCRE row(s) across {ann['rsid'].nunique()} proxies "
      f"({ann.loc[ann['in_ccre'], 'rsid'].nunique()} in at least one cCRE)")


## 9. Liver activity of the overlapping cCREs

**Why this step is necessary.** The combined Registry track is cell-type agnostic: a class of
`dELS` means "distal enhancer-like across the registry", not "active in liver". Liver activity
must be brought in separately.

### The data, and two traps in it

Files live in `liver_cCREs/chr12q/` (see that folder's `README.md`), downloaded from SCREEN on
30 July 2026 and subset to chromosome 12q — 84,656 elements each, 73 MB total instead of 2.0 GB.

**Note 1 — these are not liver-filtered subsets.** Every file contains all 2,348,854 Registry
elements. The liver call is in **column 10**, and 93–96% of rows are `Low-DNase`. Liver-active
therefore means *column 10 is an active class*, **not** membership of the file. Treating
membership as the filter would pass every element and the step would do nothing.

**Note 2 — column 11 is a data-completeness flag, not a biosample.**

| Column 11 value | Meaning |
|---|---|
| `All-data/Full-classification` | DNase + H3K4me3 + H3K27ac + CTCF assayed — can return PLS / pELS / dELS |
| `Missing-data/Partial-classification` | **DNase only** — can *only* return `Low-DNase` or `CA-only` |
| `Aggregated-Samples` | the pooled `liver.all` aggregate file |

So a non-active call in a partial-classification biosample is **not evidence of inactivity** —
the marks were never assayed. The code tracks this and reports active calls separately for
full- and partial-classification biosamples, rather than pooling them.

Because the biosample is recoverable only from the ENCFF accessions in the filename, a
**`manifest.tsv`** is required (a template with the six files pre-listed is in the folder).
Without it the notebook stops rather than guessing.


In [ ]:
# ------------------------------------------------------------------ configuration
# LIVER_CCRE_DIR, LIVER_MANIFEST and LIVER_AGGREGATE are set in section 1.
INCLUDE_LIVER_CELL_LINES = True
GROUPS = ("primary", "in_vitro_differentiated", "cell_line")   # provenance class
# Biosample categories, from the manifest's biosample_group column. Several ENCODE
# labels denote the same material ('liver tissue', 'right lobe of liver', 'left lobe of
# liver' are all bulk liver), so counts are reported per category as well as per file.
BIOSAMPLE_GROUPS = ("liver_tissue", "primary_hepatocyte",
                    "sc_derived_hepatocyte", "hepatoma_line")
# Column 10 vocabulary differs by assay coverage, so 'active' must be an explicit
# allowlist. A negated rule ("anything but Low-DNase") marks all 84,656 elements active
# in the five CHIP_ONLY files, whose vocabulary contains no Low-DNase at all.
CCRE_ACTIVE_CLASSES = {"PLS", "pELS", "dELS", "CA-CTCF", "CA-TF", "CA-only", "CA-H3K4me3"}
MARK_PREFIX  = "High-"          # CHIP_ONLY files: supporting evidence, not a cCRE class
NOT_ACTIVE   = {"Low-DNase", "Unclassified"}
FULL_FLAG    = "All-data/Full-classification"

def call_status(cls):
    """-> 'ccre_active' | 'mark_high' | 'inactive'"""
    c = str(cls)
    if c in CCRE_ACTIVE_CLASSES:      return "ccre_active"
    if c.startswith(MARK_PREFIX):     return "mark_high"
    return "inactive"

LIVER_PRIMARY_TERMS = [r"\bliver\b", r"\bhepatocyte", r"\bhepatic\b", r"hepatic stellate",
                       r"cholangiocyte", r"\bbile duct\b", r"kupffer",
                       r"lobe of liver", r"liver right lobe"]
LIVER_CELL_LINE_TERMS = [r"\bhepg2\b", r"hep\s*g2", r"huh[-\s]?7", r"hep\s*3b",
                         r"plc/prf", r"sk[-\s]?hep", r"hepatoblastoma",
                         r"hepatocellular carcinoma"]
import re
_prim = re.compile("|".join(LIVER_PRIMARY_TERMS), re.I)
_line = re.compile("|".join(LIVER_CELL_LINE_TERMS), re.I)

def classify_biosample(name):
    s = str(name)
    if _line.search(s):  return "cell_line"
    if _prim.search(s):  return "primary"
    return None


# --------------------------------------------------------------------- manifest
if not LIVER_MANIFEST.exists():
    raise RuntimeError(
        f"manifest not found at {LIVER_MANIFEST}.\n"
        "  The biosample cannot be read from the files - column 11 is a data-completeness\n"
        "  flag, not a biosample label. Resolve each ENCFF accession at\n"
        "  https://www.encodeproject.org/files/<ENCFF>/ and complete manifest.tsv.")

man = pd.read_csv(LIVER_MANIFEST, sep="\t", comment="#")
man.columns = [c.strip().lower() for c in man.columns]
if not {"file", "biosample"} <= set(man.columns):
    raise ValueError(f"manifest must have columns 'file' and 'biosample'; got {list(man.columns)}")
man["biosample"] = man["biosample"].astype("string").str.strip()
man["group"] = (man.get("group", pd.Series(index=man.index, dtype="string"))
                .astype("string").str.strip().replace({"": pd.NA}))
blank = man["biosample"].isna() | (man["biosample"] == "")
if blank.any():
    raise RuntimeError(
        "manifest.tsv has unfilled biosample entries for:\n    "
        + "\n    ".join(man.loc[blank, "file"])
        + "\n  Resolve the ENCFF accessions and complete the manifest before continuing.")
# group: use the manifest where given, else infer from the biosample name
man["group"] = [g if pd.notna(g) else classify_biosample(b)
                for g, b in zip(man["group"], man["biosample"])]
if "biosample_group" not in man.columns:
    raise RuntimeError("manifest.tsv is missing the 'biosample_group' column "
                       f"(expected one of {BIOSAMPLE_GROUPS} per row).")
print(f"manifest: {len(man)} file(s) across "
      f"{man['biosample_group'].nunique()} biosample category/ies")
print(man.groupby(['biosample_group','group']).size().to_string(), "\n")


# ------------------------------------------------------- read the per-biosample calls
def read_calls(path):
    """cCRE calls from a Registry V4 per-biosample BED. Returns start/end/acc/class/complete."""
    rows = []
    with open(path) as fh:
        for line in fh:
            f = line.rstrip("\n").split("\t")
            if len(f) < 11 or not f[0].startswith("chr"):
                continue
            rows.append((f[0], int(f[1]), int(f[2]), f[3], f[9], f[10]))
    return pd.DataFrame(rows, columns=["chrom", "start", "end", "acc", "cls", "complete"])

def calls_at(variants, calls):
    """Half-open overlap: 1-based variant p occupies 0-based [p-1, p)."""
    out = []
    for v in variants.dropna(subset=["pos_grch38"]).itertuples():
        p0 = int(v.pos_grch38) - 1
        hit = calls[(calls["chrom"] == f"chr{v.chrom}")
                    & (calls["start"] <= p0) & (p0 < calls["end"])]
        for h in hit.itertuples():
            out.append({"rsid": v.rsid, "ccre_acc_liver": h.acc,
                        "cls": h.cls, "complete": h.complete})
    return pd.DataFrame(out, columns=["rsid", "ccre_acc_liver", "cls", "complete"])

recs = []
for r in man.itertuples():
    path = LIVER_CCRE_DIR / r.file
    if not path.exists():
        raise FileNotFoundError(f"{path} listed in the manifest but not found")
    calls = read_calls(path)
    hit = calls_at(ld, calls)
    hit["file"] = r.file
    hit["biosample"] = r.biosample
    hit["group"] = r.group
    hit["biosample_group"] = getattr(r, "biosample_group", None)
    n_act = int(hit["cls"].map(lambda x: call_status(x) == "ccre_active").sum())
    comp = "FULL" if (calls["complete"].iloc[0] == FULL_FLAG) else "partial"
    print(f"  {r.biosample!r:34s} {comp:<8} {r.group or '(ungrouped)':<10} "
          f"{len(calls):,} elements, {len(hit)} proxy overlap(s), {n_act} active")
    recs.append(hit)

calls_long = pd.concat(recs, ignore_index=True)
calls_long["status"] = calls_long["cls"].map(call_status)
calls_long["is_active"] = calls_long["status"] == "ccre_active"
calls_long["mark_high"] = calls_long["status"] == "mark_high"
calls_long["full_classification"] = calls_long["complete"] == FULL_FLAG

if not INCLUDE_LIVER_CELL_LINES:
    calls_long = calls_long[calls_long["group"] != "cell_line"]

active = calls_long[calls_long["is_active"]]
if calls_long.empty:
    raise RuntimeError("no proxy overlapped any element in these files - check that the "
                       "chr12q subsets cover the locus.")

print("\n=== biosample categories in use ===")
for bg in BIOSAMPLE_GROUPS:
    sub=man[man["biosample_group"]==bg]
    if not len(sub): continue
    labels=sorted(sub["biosample"].unique())
    print(f"  {bg:<24} {len(sub):>2} file(s): {', '.join(labels)}")

print("\n=== biosamples in use (CONFIRM BEFORE PROCEEDING) ===")
for grp in GROUPS + (None,):
    names = sorted(man.loc[man["group"].isna() if grp is None else man["group"] == grp,
                           "biosample"].dropna().unique())
    if not names: continue
    tag = "  [EXCLUDED]" if grp == "cell_line" and not INCLUDE_LIVER_CELL_LINES else ""
    print(f"  {grp or 'UNGROUPED - set group in manifest.tsv':<12} ({len(names)}){tag}")
    for n in names: print(f"    - {n}")


# ------------------------------------------------- collapse to per-proxy liver status
def joinu(s):
    v = sorted({str(x) for x in s.dropna()})
    return ";".join(v) if v else None

agg = (active.groupby("rsid")
       .agg(liver_biosamples=("biosample", joinu),
            liver_classes=("cls", joinu),
            n_active_files=("biosample", "size"),          # files, not labels
            n_liver_biosamples=("biosample", "nunique"),    # distinct ENCODE labels
            n_active_biosample_groups=("biosample_group", "nunique"),
            n_liver_primary=("group", lambda s: int((s == "primary").sum())),
            n_liver_cell_lines=("group", lambda s: int((s == "cell_line").sum())),
            n_full_classification=("full_classification", lambda s: int(s.sum())))
       )
# Count FILES, not distinct ENCODE labels: several liver datasets share the label
# "liver tissue", so nunique() on the label silently collapses them.
mark = (calls_long[calls_long["mark_high"]].groupby("rsid")
        .agg(mark_high_biosamples=("biosample", joinu),
             mark_high_classes=("cls", joinu),
             n_mark_high=("biosample", "size"),
             n_mark_high_labels=("biosample", "nunique"))
        .reset_index())
grp_counts = (calls_long[calls_long["is_active"]]
              .pivot_table(index="rsid", columns="biosample_group", values="biosample",
                           aggfunc="size", fill_value=0)     # count FILES per category
              .rename(columns=lambda g: f"n_active_{g}")
              .reset_index())

# Re-running this cell must not merge the same columns into `ann` twice: pandas would
# suffix them _x/_y and the plain names would disappear. Drop any that already exist,
# so the cell is idempotent.
_incoming = [c for c in list(agg.columns) + list(mark.columns) + list(grp_counts.columns)
             if c != "rsid"]
ann = ann.drop(columns=[c for c in _incoming if c in ann.columns])

ann = ann.merge(agg, on="rsid", how="left").merge(mark, on="rsid", how="left") \
         .merge(grp_counts, on="rsid", how="left")

# If no proxy has any evidence of a given kind the merge can bring nothing back, so
# guarantee every expected column exists before it is coerced or indexed.
for _c in _incoming:
    if _c not in ann.columns:
        ann[_c] = pd.NA

for _c in [c for c in ann.columns if c.startswith("n_active_")] \
          + ["n_mark_high", "n_mark_high_labels", "n_active_biosample_groups"]:
    ann[_c] = pd.to_numeric(ann[_c], errors="coerce").fillna(0).astype(int)
for c in ("n_liver_biosamples", "n_liver_primary", "n_liver_cell_lines",
          "n_full_classification"):
    ann[c] = pd.to_numeric(ann[c], errors="coerce").fillna(0).astype(int)
ann["liver_active"] = (ann["n_liver_biosamples"] > 0) & ann["in_ccre"]
# an active call seen only in DNase-only biosamples is weaker evidence: flag it
ann["liver_active_full_only"] = (ann["n_full_classification"] > 0) & ann["liver_active"]

print(f"\nproxies in a liver-active cCRE: {ann.loc[ann['liver_active'],'rsid'].nunique()}"
      f" of {ann.loc[ann['in_ccre'],'rsid'].nunique()} in any cCRE"
      f"  ({ann.loc[ann['liver_active_full_only'],'rsid'].nunique()} with an active call in a"
      " full-classification biosample)")


# ------------------------------------------------- per-biosample detail
# Detail is printed for the lead variant and for every proxy that turns out to be
# liver-active, so the report follows the data rather than a pre-specified candidate.
_detail = [LEAD_RSID] + sorted(
    ann.loc[ann["liver_active"] & ann["passes_threshold"], "rsid"].unique().tolist())
_detail = list(dict.fromkeys(_detail))
for rs in _detail:
    sub = calls_long[calls_long["rsid"] == rs]
    print(f"\n=== {rs} ===")
    if sub.empty:
        print("  no cCRE overlap in any per-biosample file"); continue
    for r in sub.sort_values(["full_classification", "biosample"], ascending=[False, True]).itertuples():
        mark = {"ccre_active": " *ACTIVE (cCRE class)",
                "mark_high":   " *marks present",
                "inactive":    ""}[r.status]
        comp = "FULL   " if r.full_classification else "partial"
        print(f"  {r.biosample!r:34s} {comp} {r.cls:<14}{mark}")
    for name, path in LIVER_AGGREGATE.items():
        if not Path(path).exists(): continue
        a = calls_at(ld[ld["rsid"] == rs], read_calls(path))
        cls = a["cls"].iloc[0] if len(a) else "no cCRE"
        tag = "" if cls == "no cCRE" else {
            "ccre_active": "  *ACTIVE (cCRE class)",
            "mark_high":   "  *marks present",
            "inactive":    ""}[call_status(cls)]
        print(f"  {('['+name+' aggregate]'):34s} {'':7} {cls}{tag}")

    if not sub.empty:
        print("  --- by biosample category ---")
        for bg in BIOSAMPLE_GROUPS:
            g=sub[sub["biosample_group"]==bg]
            if not len(g): continue
            act=int(g["is_active"].sum()); mk=int(g["mark_high"].sum())
            print(f"    {bg:<24} {len(g):>2} file(s): {act} cCRE-active, "
                  f"{mk} with high marks")
        n_full_act = int((sub["is_active"] & sub["full_classification"]).sum())
        n_part_act = int((sub["is_active"] & ~sub["full_classification"]).sum())
        if n_full_act == 0 and n_part_act > 0:
            print("  NOTE: active only in DNase-only biosample(s); no mark-based (ELS/PLS) "
                  "support at the biosample level.")


## 9b. Liver regulatory evidence — the CRE core and its activity

Registry V4 cCREs are **anchored on DNase**: elements are defined from DNase hypersensitive
sites and then classified using H3K4me3, H3K27ac and CTCF. A biosample assayed only by ATAC
contributes nothing to element definition, and one assayed only by ChIP receives no class. Of
the 18 liver datasets, 13 have DNase and five are ChIP-only; no liver ATAC data enter the
Registry at all. Requiring Registry membership therefore imposes a DNase-derived prior that
under-represents liver.

Candidate elements are instead defined directly from peak calls, in two steps:

| Tier | Requirement |
|---|---|
| 1 — CRE core | variant inside a **replicated primary-hepatocyte ATAC peak** (peak in ≥3 of 6 donors) |
| 2 — activity | variant inside a **replicated H3K27ac peak** (≥2 of 4 donors), assessed only within the core |

Defining the core on hepatocyte ATAC matters: whole-liver ATAC peaks at this locus extend
several hundred bp further, so a variant can sit in the shoulder of a liver peak — outside the
element itself — while still overlapping a broad H3K27ac interval. Whole-liver ATAC is
therefore reported as corroboration rather than used to define the core.

Registry cCRE membership, the per-biosample liver calls and the pooled `liver.all` class are
retained as annotation (Tier 3) and reported in section 10 as a parallel funnel, so the two
approaches can be compared rather than conflated.

Every proxy is tested, not only those inside a cCRE.


In [ ]:
import glob, tempfile

_pos = (ann.dropna(subset=["pos_grch38"])[["rsid", "chrom", "pos_grch38"]]
        .drop_duplicates("rsid").sort_values("pos_grch38"))
if not len(_pos):
    raise RuntimeError("no proxy positions available - run sections 5-8 first")

def _hits(peak_file, variants):
    """rsIDs whose position falls inside a peak in `peak_file`, via bedtools intersect."""
    with tempfile.NamedTemporaryFile("w", suffix=".bed", delete=False) as vb:
        for r in variants.to_dict("records"):
            pos = int(r["pos_grch38"])
            vb.write(f'chr{r["chrom"]}\t{pos - 1}\t{pos}\t{r["rsid"]}\n')
        vpath = vb.name
    try:
        out = subprocess.run([BEDTOOLS, "intersect", "-a", vpath, "-b", str(peak_file), "-u"],
                             capture_output=True, text=True, check=True)
    finally:
        os.unlink(vpath)
    return {ln.split("\t")[3] for ln in out.stdout.splitlines() if ln.strip()}

PANEL_SUMMARY = {}
for panel, cfg in EVIDENCE_PANELS.items():
    d = Path(cfg["dir"])
    donors = sorted(glob.glob(str(d / cfg["donor_glob"]))) if d.exists() else []
    if not donors:
        raise FileNotFoundError(
            f"panel {panel!r}: no peak files matched {d / cfg['donor_glob']}. "
            "Check ATAC_ROOT and the folder layout in section 1.")
    rep = d / cfg["replicated"]

    counts = {rs: 0 for rs in _pos["rsid"]}
    for f in donors:
        for rs in _hits(f, _pos):
            counts[rs] += 1
    ann[f"n_{panel}_samples"] = pd.to_numeric(ann["rsid"].map(counts),
                                              errors="coerce").fillna(0).astype(int)

    if not rep.exists():
        raise FileNotFoundError(f"panel {panel!r}: replicated set {rep} not found")
    ann[f"{panel}_replicated"] = ann["rsid"].isin(_hits(rep, _pos))

    PANEL_SUMMARY[panel] = dict(role=cfg["role"], label=cfg["label"],
                                n_samples=len(donors),
                                sample_files=[Path(f).name for f in donors],
                                replicated_file=rep.name)
    print(f"  {panel:13s} {cfg['role']:13s} {len(donors)} sample file(s), "
          f"replicated={rep.name}: "
          f"{ann.loc[ann[f'{panel}_replicated'], 'rsid'].nunique()} proxy/proxies")

# --- tier calls ---
def _any_replicated(panels):
    out = pd.Series(False, index=ann.index)
    for p in panels:
        out = out | ann[f"{p}_replicated"].fillna(False)
    return out

# Tier 1: inside the CRE core.
ann["in_cre_core"] = ann[f"{CORE_PANEL}_replicated"].fillna(False)
# Tier 2: H3K27ac, meaningful only inside the core - masked so that the column cannot
# be read as evidence for a variant that is not in an element.
ann["mark_active"] = _any_replicated(CORE_MARK_PANELS) & ann["in_cre_core"]
ann["corroborated"] = _any_replicated(CORROBORATING_PANELS)
ann["regulatory_active"] = ann["in_cre_core"] & ann["mark_active"]

print(f"\ntier 1 in CRE core ({CORE_PANEL} replicated ATAC): "
      f"{ann.loc[ann['in_cre_core'], 'rsid'].nunique()} proxy/proxies")
print(f"tier 2 + H3K27ac ({' or '.join(CORE_MARK_PANELS)}): "
      f"{ann.loc[ann['regulatory_active'], 'rsid'].nunique()} proxy/proxies")

# Variants accessible in whole liver but OUTSIDE the hepatocyte core: peak-shoulder
# cases. Listed explicitly so the exclusion is visible rather than silent.
_shoulder = ann[ann["passes_threshold"] & ann["corroborated"] & ~ann["in_cre_core"]]
if len(_shoulder):
    print(f"\n{_shoulder['rsid'].nunique()} proxy/proxies lie in a replicated whole-liver "
          f"ATAC peak but OUTSIDE the {CORE_PANEL} core, so are not candidates:")
    print(_shoulder[["rsid", "pos_grch38", "r2_max"]
                    + [c for c in ann.columns
                       if c.startswith("n_") and c.endswith("_samples")]]
          .drop_duplicates().to_string(index=False))

# Accessible and active but absent from the Registry - the DNase-anchoring blind spot
# this section exists to detect. Reported explicitly, never silently dropped.
_blind = ann[ann["passes_threshold"] & ann["regulatory_active"] & ~ann["in_ccre"]]  # noqa
if len(_blind):
    print(f"\n{_blind['rsid'].nunique()} proxy/proxies meet both tiers but lie OUTSIDE any "
          "Registry V4 cCRE:")
    print(_blind[["rsid", "pos_grch38", "r2_max"]
                 + [c for c in ann.columns if c.endswith("_replicated")]]
          .drop_duplicates().to_string(index=False))
else:
    print("\nevery proxy meeting both tiers also lies within a Registry V4 cCRE")

_cols = ["rsid", "pos_grch38", "r2_max", "in_ccre", "liver_active",
         "in_cre_core", "mark_active", "corroborated", "regulatory_active"] \
        + [c for c in ann.columns if c.startswith("n_") and c.endswith("_samples")]
(ann[ann["passes_threshold"]][_cols].drop_duplicates()
 .sort_values("r2_max", ascending=False).head(20))


## 10. Prioritisation table

The shortlist applies all three criteria together: **R² ≥ threshold, overlap with a Registry
V4 cCRE, and activity in at least one liver biosample**. Every proxy meeting them is listed;
nothing is filtered to a pre-specified candidate.

Variants excluded by the liver criterion alone are reported separately, so the effect of that
filter is visible rather than implicit. Per-biosample detail is then printed for the lead
variant and for each shortlisted proxy.


In [ ]:
assert "liver_active" in ann.columns, "run section 9 (liver activity) first"
assert "regulatory_active" in ann.columns, "run section 9b (liver regulatory evidence) first"

# ---- primary criteria: R2 + accessible (tier 1) + H3K27ac (tier 2) ----
_r2   = ann["passes_threshold"]
short = ann[_r2 & ann["regulatory_active"]].sort_values("r2_max", ascending=False)

# ---- Registry-based funnel, reported alongside for comparison ----
reg   = ann[_r2 & ann["in_ccre"] & ann["liver_active"]]

def _n(mask): return ann.loc[mask, "rsid"].nunique()
print("=== funnel A: liver regulatory evidence (primary) ===")
print(f"  R2 >= {R2_THRESHOLD}                              {_n(_r2):>4}")
print(f"  + tier 1 in CRE core ({CORE_PANEL} ATAC, repl.)  "
      f"{_n(_r2 & ann['in_cre_core']):>4}")
print(f"  + tier 2 H3K27ac within the core        {_n(_r2 & ann['regulatory_active']):>4}")
print("\n=== funnel B: ENCODE Registry V4 (comparison) ===")
print(f"  R2 >= {R2_THRESHOLD}                              {_n(_r2):>4}")
print(f"  + within a Registry cCRE                {_n(_r2 & ann['in_ccre']):>4}")
print(f"  + Registry liver-active                 {_n(_r2 & ann['in_ccre'] & ann['liver_active']):>4}")

_a, _b = set(short["rsid"]), set(reg["rsid"])
print(f"\nconvergence: {len(_a & _b)} variant(s) selected by both, "
      f"{len(_a - _b)} only by A, {len(_b - _a)} only by B")
if _a - _b: print(f"  only by A (missed by the Registry): {sorted(_a - _b)}")
if _b - _a: print(f"  only by B (no replicated ATAC/H3K27ac): {sorted(_b - _a)}")

print(f"\n=== prioritisation shortlist: R2 >= {R2_THRESHOLD}, within a replicated "
      f"{CORE_PANEL} ATAC peak and H3K27ac-marked "
      f"({short['rsid'].nunique()} variant(s)) ===")
_cols = ["rsid","pos_grch38","distance_bp","r2_ldlink","r2_ensembl",
         "in_cre_core","mark_active","corroborated"] \
        + [c for c in short.columns if c.startswith("n_") and c.endswith("_samples")] \
        + ["in_ccre","ccre_accession","ccre_class","ccre_class_liver_all","liver_active"] \
        + [c for c in short.columns if c.startswith("n_active_")] \
        + ["n_mark_high","n_full_classification"]
print(short[[c for c in _cols if c in short.columns]].to_string(index=False))

dropped = ann[_r2 & (ann["in_ccre"] | ann["in_cre_core"] | ann["corroborated"])
              & ~ann["regulatory_active"]]
if len(dropped):
    print(f"\n{dropped['rsid'].nunique()} variant(s) had some regulatory evidence but did "
          "not meet both tiers:")
    _dc = ["rsid","pos_grch38","r2_max","in_ccre","ccre_class","liver_active",
           "in_cre_core","mark_active","corroborated"] \
          + [c for c in dropped.columns if c.startswith("n_") and c.endswith("_samples")]
    print(dropped[[c for c in _dc if c in dropped.columns]]
          .drop_duplicates().to_string(index=False))

print("\n=== lead variant and shortlisted proxies ===")
# The LD source may report the lead under a merged dbSNP identifier, so find it by
# GRCh38 position and fall back to the configured rsID only if the position is absent.
_lead_rows = ann[ann["pos_grch38"] == lead["pos"]]
LEAD_IN_SET = _lead_rows["rsid"].iloc[0] if len(_lead_rows) else LEAD_RSID
if LEAD_IN_SET != LEAD_RSID:
    print(f"  note: the LD source reports the lead variant as {LEAD_IN_SET}, not "
          f"{LEAD_RSID}, at the same GRCh38 position "
          f"(chr{lead['chrom']}:{lead['pos']:,}). This is usually a merged dbSNP "
          "identifier - confirm which rsID to quote before reporting.")
for rs in [LEAD_IN_SET] + [r for r in short["rsid"].unique() if r != LEAD_IN_SET]:
    rows = ann[ann["rsid"] == rs]
    if rows.empty:
        print(f"  {rs}: no record at R2 >= {R2_THRESHOLD} within "
              f"+/-{WINDOW_KB} kb of the lead"); continue
    r = rows.iloc[0]
    r2s = (f"LDlink {r['r2_ldlink']:.3f}" if pd.notna(r["r2_ldlink"]) else "LDlink n/a")
    r2e = (f"Ensembl {r['r2_ensembl']:.3f}" if pd.notna(r["r2_ensembl"]) else "Ensembl n/a")
    print(f"  {rs}: chr{r['chrom']}:{int(r['pos_grch38']):,}  {r2s}  {r2e}  "
          f"passes={bool(r['passes_threshold'])}")
    for _, h in rows.iterrows():
        if pd.notna(h["ccre_accession"]):
            print(f"      cCRE {h['ccre_accession']}  class={h['ccre_class']}  "
                  f"[{int(h['ccre_start']):,}-{int(h['ccre_end']):,}]")
        else:
            print("      no overlapping cCRE in this registry version")


---
Look up of **rs59666428** in dbSNP (https://www.ncbi.nlm.nih.gov/snp/?term=rs59666428) shows it has been merged into **rs11183620**.

## 11. Optional — was each proxy actually tested in the GWAS?

The exploratory GWAS was genotyped on an array, so most LD proxies were never tested. This
join makes that explicit and prevents a proxy being described as showing association when
it carries no association statistic. Point `GWAS_FILE` at the PLINK `.assoc.logistic`
output; leave as `None` to skip.

Note the summary statistics are on **GRCh37** and identify some variants by Illumina probe
ID (`exm...`) rather than rsID, so the join is on rsID only and will under-match.


In [ ]:
GWAS_FILE = None   # e.g. Path("STOPAH_GWAS.finalqc_not_gender_adjusted.maf0.01.assoc.logistic")

if GWAS_FILE and Path(GWAS_FILE).exists():
    g = pd.read_csv(GWAS_FILE, sep=r"\s+")
    g = g.rename(columns={"SNP": "rsid"})[["rsid", "A1", "OR", "L95", "U95", "P"]]
    ann = ann.merge(g, on="rsid", how="left")
    ann["tested_in_gwas"] = ann["P"].notna()
    n = int(ann.loc[ann["passes_threshold"], "tested_in_gwas"].sum())
    tot = int(ann["passes_threshold"].sum())
    print(f"{n}/{tot} threshold-passing proxies were tested in the exploratory GWAS")
    t = ann[ann["passes_threshold"] & ann["tested_in_gwas"]].sort_values("P")
    if len(t):
        print("\nproxies with association statistics, best first:")
        print(t[["rsid","pos_grch38","r2_max","OR","L95","U95","P","ccre_class"]]
              .to_string(index=False))
    unt = ann[ann["passes_threshold"] & ~ann["tested_in_gwas"] & ann["in_ccre"]]
    if len(unt):
        print(f"\n{unt['rsid'].nunique()} cCRE-overlapping proxy/proxies have NO association\n"
              "statistic - these must not be described as 'showing association' in the text:")
        print(sorted(unt["rsid"].unique()))
else:
    ann["tested_in_gwas"] = pd.NA
    print("skipped - set GWAS_FILE to enable")


## 12. Proxy summary table and BED track

Two flat exports, one row per proxy — no exploded proxy×cCRE rows:

- **`SLC38A4_proxies_summary.csv`** — R² against the lead variant from both sources, plus a
  single `overlaps_ccre` True/False column, and `passes_r2_threshold` (the threshold
  value is recorded in the provenance file rather than baked into a column name). Where a proxy falls in more than one cCRE the
  accessions and classes are collapsed into semicolon-separated fields, so the row count
  equals the proxy count.
- **`SLC38A4_LD_proxies.bed`** — loadable as a UCSC custom track or in IGV. Positions are
  converted to BED's 0-based half-open convention (a SNP at 1-based *p* becomes
  `[p-1, p)`), the score column carries R²×1000 so the browser can shade by LD, and
  `itemRgb` colours each variant by whether it clears the R² threshold and whether it hits
  a cCRE.

- **`SLC38A4_cCRE_prioritisation.bed`** — the prioritisation set only: proxies with
  **R² ≥ threshold, overlapping a cCRE, and that cCRE active in at least one liver
  biosample**, matching the rows in `SLC38A4_cCRE_prioritisation.csv`. The cCRE class is
  carried into each feature name (e.g. `rsID_dELS_r2=0.85`). Because every feature here
  satisfies all criteria, colour encodes **cCRE class** rather than pass/fail — red PLS,
  orange pELS, yellow dELS, blue CTCF-only, approximating SCREEN's own palette — so the
  track stays informative instead of uniformly one colour.

- **`SLC38A4_cCRE_notLiverActive.bed`** — variants that met the R² and cCRE criteria but
  whose cCRE is not liver-active. Written separately rather than silently dropped, so the
  effect of the liver filter is auditable.

A companion `SLC38A4_cCREs_V4_window.bed` is also written, so the proxies and the cCREs can
be loaded together to reproduce the Figure 5A view.


In [ ]:
# ---------------------------------------------- one row per proxy
def join_unique(s):
    vals = [str(v) for v in s.dropna().unique()]
    return ";".join(vals) if vals else None

# Built from whatever columns `ann` actually carries, so that renaming a field
# upstream cannot silently drop it - or crash this cell after the whole pipeline
# has run. Anything genuinely indispensable is checked separately below.
FIRST_COLS = ["rsid_alt", "allele_match", "chrom", "pos_grch38", "alleles", "maf",
              "distance_bp", "r2_ldlink", "r2_ensembl", "r2_max", "passes_threshold",
              "threshold_is_source_dependent",
              "tested_in_gwas", "P", "OR", "L95", "U95"]
JOIN_COLS  = ["ccre_accession", "ccre_class", "ccre_class_liver_all",
              "liver_biosamples", "liver_classes", "mark_high_biosamples",
              "mark_high_classes"]
MAX_COLS   = ["liver_active", "in_cre_core", "mark_active", "corroborated",
              "regulatory_active",
              "n_active_files", "n_active_biosample_groups",
              "n_mark_high", "n_full_classification",
              "n_liver_primary", "n_liver_cell_lines"] \
             + [c for c in ann.columns if c.startswith("n_active_")] \
             + [c for c in ann.columns
                if c.endswith("_samples") or c.endswith("_replicated")]

REQUIRED = ["chrom", "pos_grch38", "r2_max", "passes_threshold",
            "ccre_accession", "liver_active"]
_missing = [c for c in REQUIRED if c not in ann.columns]
if _missing:
    raise KeyError(f"ann is missing required column(s) {_missing}; "
                   f"available: {sorted(ann.columns)}")

agg = {c: "first"     for c in FIRST_COLS if c in ann.columns}
agg.update({c: join_unique for c in JOIN_COLS if c in ann.columns})
agg.update({c: "max"       for c in dict.fromkeys(MAX_COLS) if c in ann.columns})
_dropped = [c for c in FIRST_COLS + JOIN_COLS + MAX_COLS if c not in ann.columns]
if _dropped:
    print(f"  note: not summarised (absent from ann): {sorted(set(_dropped))}")

summary = (ann.groupby("rsid", as_index=False, dropna=False).agg(agg))

summary["overlaps_ccre"] = summary["ccre_accession"].notna()
summary["n_ccre"] = summary["ccre_accession"].fillna("").apply(
    lambda s: len([x for x in s.split(";") if x]))
# By position, not identifier: the LD source may report the lead under a merged rsID.
summary["is_lead"] = summary["pos_grch38"] == lead["pos"]
summary["liver_active"] = summary["liver_active"].fillna(False).astype(bool)
for cc in ("n_liver_primary", "n_liver_cell_lines"):
    summary[cc] = pd.to_numeric(summary[cc], errors="coerce").fillna(0).astype(int)

summary = summary.rename(columns={
    "distance_bp": "distance_from_lead_bp",
    "passes_threshold": "passes_r2_threshold",
    "ccre_accession": "ccre_accessions",
    "ccre_class": "ccre_classes",
    "P": "gwas_P", "OR": "gwas_OR", "L95": "gwas_L95", "U95": "gwas_U95",
})

PREFERRED = ["rsid", "rsid_alt", "chrom", "pos_grch38", "alleles", "allele_match",
             "maf", "distance_from_lead_bp",
             "r2_ldlink", "r2_ensembl", "r2_max", "passes_r2_threshold",
             "threshold_is_source_dependent", "overlaps_ccre", "n_ccre",
             "ccre_accessions", "ccre_classes", "ccre_class_liver_all",
             "in_cre_core", "mark_active", "corroborated", "regulatory_active",
             "n_PHH_samples", "PHH_replicated",
             "n_ENCODE_liver_samples", "ENCODE_liver_replicated",
             "n_H3K27ac_samples", "H3K27ac_replicated",
             "liver_active", "n_active_files", "n_active_biosample_groups",
             "n_mark_high", "n_full_classification",
             "n_liver_primary", "n_liver_cell_lines",
             "liver_biosamples", "liver_classes",
             "mark_high_biosamples", "mark_high_classes", "is_lead",
             "tested_in_gwas", "gwas_OR", "gwas_L95", "gwas_U95", "gwas_P"]
cols = [c for c in PREFERRED if c in summary.columns]
cols += [c for c in summary.columns if c not in cols]   # keep anything unlisted
summary = summary[cols].sort_values("r2_max", ascending=False).reset_index(drop=True)

assert summary["rsid"].is_unique, "summary should be one row per proxy"
assert len(summary) == ann["rsid"].nunique()

summary.to_csv(OUTDIR / "SLC38A4_proxies_summary.csv", index=False)
print(f"SLC38A4_proxies_summary.csv: {len(summary)} proxies, "
      f"{int(summary['overlaps_ccre'].sum())} overlapping a cCRE "
      f"({int(summary['liver_active'].sum())} liver-active), "
      f"{int(summary['passes_r2_threshold'].sum())} above the R2 threshold")


# ---------------------------------------------- BED export
# BED is 0-based half-open; variant positions are 1-based.
RGB = {("pass", "ccre"): "220,20,60",     # crimson - clears R2 and hits a cCRE
       ("pass", "none"): "30,90,200",     # blue    - clears R2 only
       ("fail", "ccre"): "255,140,0",     # orange  - hits a cCRE but below R2
       ("fail", "none"): "150,150,150"}   # grey    - neither
LEAD_RGB = "0,0,0"

# cCRE class palette, approximating SCREEN's own colours, for tracks where every
# feature already clears the R2 threshold and hits a cCRE - there, class is the
# informative dimension rather than pass/fail status.
# Two vocabularies reach this cell: the UCSC track returns descriptive names
# ("Distal enhancer", "CA", "TF"), while the Registry files use codes (dELS, CA-only).
# Both are mapped so the track is coloured either way; anything unmapped renders grey,
# which is cosmetic only and never affects the shortlist.
CLASS_RGB = {"PLS": "255,0,0",   "Promoter":          "255,0,0",
             "pELS": "255,167,0", "Proximal enhancer": "255,167,0",
             "dELS": "255,205,0", "Distal enhancer":   "255,205,0",
             "CA-CTCF": "0,176,240", "CTCF": "0,176,240",
             "CA-TF": "190,120,220", "TF":   "190,120,220",
             "CA-only": "110,190,110", "CA":  "110,190,110",
             "CA-H3K4me3": "255,170,170",
             "Low-DNase": "230,230,230"}
CLASS_RGB_DEFAULT = "150,150,150"

def write_bed(df, path, track_name, description, name_with_ccre=False,
              color_by="status"):
    rows = []
    # dict iteration, not itertuples: column names here need not be valid identifiers
    for r in df.dropna(subset=["pos_grch38"]).to_dict("records"):
        pos = int(r["pos_grch38"])
        r2 = r.get("r2_max")
        score = 0 if pd.isna(r2) else max(0, min(1000, int(round(float(r2) * 1000))))
        if bool(r["is_lead"]):
            rgb = LEAD_RGB
        elif color_by == "ccre_class":
            first = str(r.get("ccre_classes") or "").split(";")[0].strip()
            rgb = CLASS_RGB.get(first, CLASS_RGB_DEFAULT)
        else:
            rgb = RGB[("pass" if bool(r["passes_r2_threshold"]) else "fail",
                       "ccre" if bool(r["overlaps_ccre"]) else "none")]
        r2txt = 0 if pd.isna(r2) else round(float(r2), 3)
        cls = r.get("ccre_classes")
        label = (f"{r['rsid']}_{cls}_r2={r2txt}"
                 if name_with_ccre and isinstance(cls, str) and cls
                 else f"{r['rsid']}_r2={r2txt}")
        rows.append((f"chr{r['chrom']}", pos - 1, pos, label, score, ".",
                     pos - 1, pos, rgb))
    rows.sort(key=lambda x: (x[0], x[1]))
    with open(path, "w") as fh:
        fh.write(f'track name="{track_name}" description="{description}" '
                 'itemRgb="On" visibility=pack useScore=0\n')
        for row in rows:
            fh.write("\t".join(map(str, row)) + "\n")
    return len(rows)

n_bed = write_bed(
    summary, OUTDIR / "SLC38A4_LD_proxies.bed",
    f"SLC38A4 LD proxies ({POP})",
    f"LD proxies of {LEAD_RSID}; score=R2x1000; crimson=R2>={R2_THRESHOLD} and in cCRE, "
    f"blue=R2>={R2_THRESHOLD} only, orange=in cCRE only, grey=neither, black=lead variant")
print(f"SLC38A4_LD_proxies.bed: {n_bed} features (GRCh38/hg38)")

# --- the prioritisation set: R2 >= threshold AND overlapping a cCRE ---
# Same definition as section 10: R2 + accessible + H3K27ac. Registry membership is
# annotation, not a criterion, so it does not appear here.
summary.to_csv(OUTDIR / "SLC38A4_proxies_summary.csv", index=False)
print(f"SLC38A4_proxies_summary.csv: {len(summary)} proxies (one row each)")

prioritised = summary[summary["passes_r2_threshold"] & summary["regulatory_active"]]
if len(prioritised) == 0:
    print(f"\nNOTE: no proxy has R2 >= {R2_THRESHOLD}, overlaps a cCRE and is liver-active"
          " - SLC38A4_cCRE_prioritisation.bed not written.")
else:
    n_sub = write_bed(
        prioritised, OUTDIR / "SLC38A4_cCRE_prioritisation.bed",
        f"SLC38A4 prioritised proxies (R2>={R2_THRESHOLD}, liver-active cCRE)",
        f"LD proxies of {LEAD_RSID} with R2>={R2_THRESHOLD} ({POP}) in a cCRE "
        f"({CCRE_TRACK}) active in >=1 liver biosample; score=R2x1000; colour=cCRE class "
        f"(red PLS, orange pELS, yellow dELS, blue CTCF-only, black lead variant)",
        name_with_ccre=True, color_by="ccre_class")
    print(f"SLC38A4_cCRE_prioritisation.bed: {n_sub} feature(s) "
          f"- R2 >= {R2_THRESHOLD}, in a cCRE, liver-active")
    print("   classes:", ", ".join(
        f"{k}={v}" for k, v in prioritised["ccre_classes"].value_counts().items()))

# the same set without the liver requirement, for comparison in review
no_liver = summary[summary["passes_r2_threshold"]
                   & (summary["overlaps_ccre"] | summary["in_cre_core"]
                      | summary["corroborated"])
                   & ~summary["regulatory_active"]]
if len(no_liver):
    write_bed(no_liver, OUTDIR / "SLC38A4_cCRE_notLiverActive.bed",
              f"SLC38A4 proxies in non-liver-active cCREs (R2>={R2_THRESHOLD})",
              "met the R2 and cCRE criteria but the cCRE is not active in any liver "
              "biosample; retained for transparency",
              name_with_ccre=True, color_by="ccre_class")
    print(f"SLC38A4_cCRE_notLiverActive.bed: {len(no_liver)} feature(s) "
          "(excluded from the shortlist; kept for transparency)")

# companion cCRE track for the same window
cb = ccre.dropna(subset=["start", "end"]).sort_values("start")
with open(OUTDIR / "SLC38A4_cCREs_V4_window.bed", "w") as fh:
    fh.write(f'track name="cCRE V4 ({CCRE_TRACK})" '
             f'description="{CCRE_LABEL}" visibility=pack\n')
    for r in cb.itertuples():
        name = f"{r.ccre_accession}_{r.ccre_class}" if r.ccre_class else str(r.ccre_accession)
        fh.write(f"{r.chrom}\t{int(r.start)}\t{int(r.end)}\t{name}\t0\t.\n")
print(f"SLC38A4_cCREs_V4_window.bed: {len(cb)} elements")

print(f"\n--- load in UCSC: My Data > Custom Tracks > upload the .bed files "
      f"(assembly {GENOME}) ---")
summary.head(25)


### 12c. High-LD-only track (R² ≥ threshold, no cCRE requirement)

**`SLC38A4_LD_proxies_R2ge0.7.bed`** — just the proxies that clear the R² threshold
against the lead variant, independent of whether they overlap a cCRE (unlike
`SLC38A4_cCRE_prioritisation.bed`, which additionally requires liver-active
regulatory evidence). Same BED conventions and score encoding as
`SLC38A4_LD_proxies.bed`, colour restricted to crimson (in a cCRE) / blue (not in a
cCRE) / black (lead variant), since every feature here already clears R² — for a
main-figure LD track that shows only the high-LD set rather than the full
background of tested proxies.

In [ ]:
# --- high-LD-only track: R2 >= threshold, no cCRE/liver requirement ---
high_ld = summary[summary["passes_r2_threshold"]].copy()
n_hl = write_bed(
    high_ld, OUTDIR / "SLC38A4_LD_proxies_R2ge0.7.bed",
    f"SLC38A4 high-LD proxies ({POP}, R2>={R2_THRESHOLD})",
    f"{len(high_ld)} variants at R2>={R2_THRESHOLD} with lead {LEAD_RSID} ({POP}); "
    f"score=R2x1000; crimson=in a cCRE, blue=not in a cCRE, black=lead variant")
print(f"SLC38A4_LD_proxies_R2ge0.7.bed: {n_hl} feature(s) "
      f"- R2 >= {R2_THRESHOLD} against {LEAD_RSID} ({POP}), cCRE status not required")

## 12b. Table — per-dataset evidence at the prioritised variant(s)

One row per dataset per prioritised variant, across all three evidence layers: the individual
ATAC and H3K27ac samples with the coordinates of the peak covering the variant, and each of the
18 Registry V4 liver datasets with its classification. This is what substantiates statements of
the form "active in *n* of 18 datasets" and makes the provenance of the reported class explicit.


In [ ]:
# ---- per-dataset evidence at each prioritised variant ----
def _covering(peak_file, chrom, pos):
    """Intervals in `peak_file` that contain `pos` (1-based), via bedtools."""
    with tempfile.NamedTemporaryFile("w", suffix=".bed", delete=False) as vb:
        vb.write(f"chr{chrom}\t{pos - 1}\t{pos}\tv\n")
        vpath = vb.name
    try:
        out = subprocess.run([BEDTOOLS, "intersect", "-a", vpath, "-b", str(peak_file),
                              "-wa", "-wb"], capture_output=True, text=True, check=True)
    finally:
        os.unlink(vpath)
    hits = []
    for ln in out.stdout.splitlines():
        f = ln.split("\t")
        if len(f) >= 7:
            hits.append((int(f[5]), int(f[6])))
    return hits

rows = []
for rs in short["rsid"].unique():
    _r = ann[ann["rsid"] == rs].iloc[0]
    chrom, pos = str(_r["chrom"]), int(_r["pos_grch38"])

    for panel, cfg in EVIDENCE_PANELS.items():
        d = Path(cfg["dir"])
        files = sorted(glob.glob(str(d / cfg["donor_glob"])))
        files += [str(d / cfg["replicated"])]
        for f in files:
            is_rep = Path(f).name == cfg["replicated"]
            hits = _covering(f, chrom, pos)
            rows.append(dict(
                rsid=rs, pos_grch38=pos,
                evidence_layer=("replicated set" if is_rep else "individual sample"),
                panel=panel, panel_role=cfg["role"], dataset=Path(f).name,
                covers_variant=bool(hits),
                interval=(f"chr{chrom}:{hits[0][0]:,}-{hits[0][1]:,}" if hits else None),
                interval_bp=(hits[0][1] - hits[0][0]) if hits else None,
                call=None, biosample=None, biosample_group=None, assay_coverage=None))

    sub = calls_long[calls_long["rsid"] == rs]
    for r in sub.to_dict("records"):
        rows.append(dict(
            rsid=rs, pos_grch38=pos, evidence_layer="ENCODE Registry V4",
            panel="Registry V4", panel_role="annotation",
            dataset=r.get("file"), covers_variant=True,
            interval=None, interval_bp=None,
            call=r.get("cls"), biosample=r.get("biosample"),
            biosample_group=r.get("biosample_group"),
            assay_coverage=("FULL" if r.get("full_classification") else "partial")))

    agg_cls = _r.get("ccre_class_liver_all")
    if pd.notna(agg_cls):
        rows.append(dict(rsid=rs, pos_grch38=pos, evidence_layer="ENCODE Registry V4",
                         panel="Registry V4", panel_role="annotation",
                         dataset="liver.all (pooled aggregate)", covers_variant=True,
                         interval=None, interval_bp=None, call=agg_cls,
                         biosample="all liver biosamples", biosample_group=None,
                         assay_coverage="pooled signal"))

per_dataset = pd.DataFrame(rows)
if per_dataset.empty:
    print("no prioritised variant - per-dataset table not written")
else:
    _order = ["rsid","pos_grch38","evidence_layer","panel","panel_role","dataset",
              "biosample","biosample_group","assay_coverage","covers_variant",
              "interval","interval_bp","call"]
    per_dataset = per_dataset[_order]
          f"for {per_dataset['rsid'].nunique()} prioritised variant(s)")
    for rs, g in per_dataset.groupby("rsid"):
        ind = g[g["evidence_layer"] == "individual sample"]
        reg = g[(g["evidence_layer"] == "ENCODE Registry V4") & (g["panel_role"] == "annotation")]
        reg = reg[reg["dataset"].astype(str).str.endswith(".bed")]
        print(f"\n  {rs}:")
        for panel, gg in ind.groupby("panel"):
            print(f"    {panel:13s} {int(gg['covers_variant'].sum())}/{len(gg)} samples")
        if len(reg):
            act = reg["call"].isin(CCRE_ACTIVE_CLASSES).sum()
            print(f"    Registry V4   {act}/{len(reg)} datasets with an active class")

per_dataset.head(20)


## 13. Output

In [ ]:
ld.to_csv(OUTDIR / "SLC38A4_LD_proxies_GRCh38.csv", index=False)
ann.to_csv(OUTDIR / "SLC38A4_proxies_cCRE_V4.csv", index=False)
short.to_csv(OUTDIR / "SLC38A4_cCRE_prioritisation.csv", index=False)

provenance = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "assembly": "GRCh38/hg38 throughout (no liftover)",
    "lead_variant": {"rsid": LEAD_RSID, "chrom": lead["chrom"], "pos_grch38": lead["pos"]},
    "window_kb": WINDOW_KB,
    "r2_threshold": R2_THRESHOLD,
    "population": POP,
    "ld_sources": {
        "ldlink": {"endpoint": f"{LDLINK}/ldproxy", "population": POP,
                   "genome_build": "grch38", "used": bool(LDLINK_TOKEN),
                   "n_proxies": int(len(ld_a))},
        "ensembl": {"endpoint": f"{ENSEMBL}/ld/human",
                    "population": f"1000GENOMES:phase_3:{POP}",
                    "used": True, "n_proxies": int(len(ld_b))},
    },
    "ccre_registry": {"source": "UCSC REST API", "genome": GENOME,
                      "track": CCRE_TRACK, "longLabel": CCRE_LABEL,
                      "n_elements_in_window": int(len(ccre))},
    "liver_activity_filter": {
        "applied": True,
        "source": "SCREEN per-biosample Registry V4 files (chr12q subsets)",
        "ccre_dir": str(LIVER_CCRE_DIR),
        "manifest": str(LIVER_MANIFEST),
        "biosamples": man[["file","biosample","group","biosample_group",
                           "classification"]].to_dict("records"),
        # An explicit allowlist, not a negated rule: the column-10 vocabulary differs
        # by assay coverage, so "anything but Low-DNase" would call every element in a
        # ChIP-only file active.
        "active_definition": {
            "ccre_active": sorted(CCRE_ACTIVE_CLASSES),
            "mark_high":   f"{MARK_PREFIX}* labels in ChIP-only files "
                           "(supporting evidence, not a cCRE class)",
            "not_active":  sorted(NOT_ACTIVE),
        },
        "aggregate_crosschecks": {k: str(v) for k, v in LIVER_AGGREGATE.items()},
        "include_liver_cell_lines": INCLUDE_LIVER_CELL_LINES,
        "region_subset": "chr12q (end > 37,200,000, GRCh38)",
        "regulatory_evidence_filter": {
            "tier1_core_panel": CORE_PANEL,
            "tier2_mark_panels": list(CORE_MARK_PANELS),
            "corroborating_panels": list(CORROBORATING_PANELS),
            "definition": "the CRE core is a replicated ATAC peak in the core panel; a "
                          "variant must lie inside that core, and H3K27ac is assessed only "
                          "within it. Whole-liver ATAC corroborates but does not define the "
                          "core. Registry V4 membership is annotation only.",
            "panels": PANEL_SUMMARY,
        },
        "primary_terms": LIVER_PRIMARY_TERMS,
        "cell_line_terms": LIVER_CELL_LINE_TERMS,
        "matched_biosamples": sorted(man["biosample"].dropna().unique().tolist()),
        "n_liver_active_proxies": int(ann.loc[ann["liver_active"], "rsid"].nunique()),
        "note": ("the combined cCRE registry track is cell-type agnostic; liver activity "
                 "was established from per-biosample classifications"),
    },
    "intersection_method": "bedtools intersect",
    "bedtools": {"path": str(BEDTOOLS), "version": BEDTOOLS_VERSION,
                 "verified_against": "independent half-open interval calculation"},
    "counts": {"proxies_in_window": int(len(ld)),
               "proxies_passing_r2": int(ld["passes_threshold"].sum()),
               "proxies_in_ccre": int(ann.loc[ann["in_ccre"], "rsid"].nunique()),
               "proxies_in_liver_active_ccre":
                   int(ann.loc[ann["liver_active"], "rsid"].nunique()),
               "shortlist": int(short["rsid"].nunique()),
               "threshold_source_dependent": int(ld["threshold_is_source_dependent"].sum())},
    "bed_convention": "0-based half-open; 1-based SNP position p exported as [p-1, p)",
    "package_versions": {"python": sys.version.split()[0], "pandas": pd.__version__,
                         "requests": requests.__version__},
}
(OUTDIR / "SLC38A4_cCRE_provenance.json").write_text(json.dumps(provenance, indent=2))

print(f"shortlist: {short['rsid'].nunique()} variant(s) | "
      f"proxies in window: {len(ld)} | cCREs in window: {len(ccre)}")
print("written: SLC38A4_LD_proxies_GRCh38.csv, SLC38A4_proxies_cCRE_V4.csv,\n"
      "         SLC38A4_cCRE_prioritisation.csv, SLC38A4_proxies_summary.csv,\n"
      "         SLC38A4_LD_proxies.bed, SLC38A4_LD_proxies_R2ge0.7.bed,\n"
      "         SLC38A4_cCRE_prioritisation.bed,\n"
      "         SLC38A4_cCRE_notLiverActive.bed (if any),\n"
      "         SLC38A4_cCREs_V4_window.bed,\n"
      "         SLC38A4_cCRE_provenance.json")

short[["rsid","pos_grch38","distance_bp","r2_ldlink","r2_ensembl",
       "ccre_accession","ccre_class"]]


## Table S14 — per-variant LD + cCRE evidence used to prioritise rs4994910

One row per proxy at R² ≥ threshold (48 of the 1,483 positions tested, including the lead
variant for reference), reporting both prioritisation criteria side by side: the primary
hepatocyte-ATAC + liver-H3K27ac criterion used in the manuscript, and the ENCODE Registry V4
comparison criterion (Section 5.3, Supplementary Methods). Built from `summary` directly, so it
cannot drift from the BED tracks/prioritisation exports above.

In [ ]:
# ---- Table S14 -- per-variant LD + cCRE evidence used to prioritise rs4994910 ----
st14 = summary[summary["passes_r2_threshold"]].copy()

_RENAME14 = {
    "rsid": "rsID", "rsid_alt": "alternate/merged rsID (if any)",
    "chrom": "chromosome", "pos_grch38": "position (GRCh38)",
    "alleles": "alleles", "maf": "MAF (GBR)",
    "distance_from_lead_bp": "distance from lead variant (bp)",
    "r2_ldlink": "R2 (LDlink)", "r2_ensembl": "R2 (Ensembl)",
    "r2_max": "R2 (max of the two sources)",
    "allele_match": "allele match between LD sources",
    "threshold_is_source_dependent": "R2 threshold pass/fail differs between LD sources",
    "is_lead": "is lead variant",
    "in_cre_core": "within replicated hepatocyte ATAC-seq peak",
    "n_PHH_samples": "primary hepatocyte donors with ATAC-seq peak (of 6)",
    "PHH_replicated": "replicated primary hepatocyte ATAC-seq peak (>=3/6 donors)",
    "mark_active": "within replicated liver H3K27ac peak",
    "n_H3K27ac_samples": "liver H3K27ac experiments with peak (of 4)",
    "H3K27ac_replicated": "replicated liver H3K27ac peak (>=2/4 experiments)",
    "regulatory_active": "meets primary prioritisation criteria",
    "corroborated": "within replicated whole-liver ATAC-seq peak (corroboration only)",
    "n_ENCODE_liver_samples": "whole-liver ATAC-seq experiments with peak (of 6)",
    "ENCODE_liver_replicated": "replicated whole-liver ATAC-seq peak (>=3/6, 50% reciprocal overlap)",
    "overlaps_ccre": "within an ENCODE Registry V4 cCRE",
    "n_ccre": "number of overlapping Registry cCREs",
    "ccre_accessions": "Registry cCRE accession(s)", "ccre_classes": "Registry cCRE class(es)",
    "ccre_class_liver_all": "Registry cCRE class (liver.all aggregate)",
    "liver_active": "active in >=1 of 18 Registry liver datasets",
    "n_active_files": "number of active Registry liver datasets",
    "n_active_biosample_groups": "number of active Registry liver biosample groups",
    "n_active_liver_tissue": "active in liver tissue datasets (n)",
    "n_active_primary_hepatocyte": "active in primary hepatocyte datasets (n)",
    "n_active_sc_derived_hepatocyte": "active in stem-cell-derived hepatocyte datasets (n)",
    "n_active_hepatoma_line": "active in hepatoma cell-line datasets (n)",
    "liver_biosamples": "active Registry liver biosample(s)",
    "liver_classes": "Registry class(es) in active liver biosample(s)",
    "n_mark_high": "ChIP-only datasets with High- mark evidence (n)",
    "n_full_classification": "liver datasets with full assay classification (n)",
    "mark_high_biosamples": "ChIP-only biosample(s) with High- mark evidence",
    "mark_high_classes": "High- mark class(es)",
}
st14 = st14.rename(columns=_RENAME14)
st14 = st14[list(_RENAME14.values())].sort_values(
    "R2 (max of the two sources)", ascending=False).reset_index(drop=True)

# Annotate the lead variant's manuscript rsID explicitly: LDlink/Ensembl report this position
# under rs59666428, not rs11183620 -- confirmed via dbSNP to be the identical GRCh38 position
# (chr12:46,818,587). Not captured by the notebook's own rsid_alt column, so recorded here.
st14["note"] = ""
st14.loc[st14["is lead variant"], "note"] = (
    "Lead GWAS variant (rs11183620); LDlink/Ensembl report this position under rsID "
    "rs59666428 -- confirmed via dbSNP to be the same variant (identical GRCh38 position "
    "chr12:46,818,587).")
st14.loc[st14["rsID"] == "rs4994910", "note"] = (
    "Prioritised variant carried forward for functional assessment (Fig. 5, Table S15).")

n_total = len(summary); n_r2 = len(st14)
n_reg_ccre = int(st14["within an ENCODE Registry V4 cCRE"].sum())
n_reg_active = int((st14["within an ENCODE Registry V4 cCRE"]
                    & st14["active in >=1 of 18 Registry liver datasets"]).sum())
n_primary = int(st14["meets primary prioritisation criteria"].sum())

ST14_PATH = "Table_S14_LD_cCRE_rs4994910.xlsx"
_readme14 = pd.DataFrame({"Table S14 - Per-variant linkage disequilibrium and "
                          "candidate cis-regulatory element (cCRE) evidence used to prioritise rs4994910": [
    f"Variants in high linkage disequilibrium (R2 >= 0.7, 1000 Genomes GBR panel) with the lead "
    f"GWAS variant rs11183620 at the SLC38A4 locus, retrieved from LDlink LDproxy and cross-checked "
    f"against the Ensembl REST LD endpoint (union of both sources; {n_total} proxy positions "
    f"tested, {n_r2} met the R2 threshold and are listed here).",
    "Two parallel regulatory-evidence criteria are reported for each variant: (1) the primary "
    "criterion used in the manuscript -- overlap with a replicated ATAC-seq peak in primary human "
    "hepatocytes (>=3/6 donors) AND a replicated liver H3K27ac ChIP-seq peak (>=2/4 ENCODE "
    "experiments); and (2) a comparison criterion using the ENCODE Registry of candidate "
    "cis-regulatory elements V4 -- overlap with a Registry cCRE, active (PLS/pELS/dELS/CA-CTCF/"
    "CA-TF/CA-only/CA-H3K4me3) in >=1 of 18 curated liver biosamples. Both approaches converge on "
    "the same variant, rs4994910.",
    f"Of {n_r2} variants at R2>=0.7: {n_reg_ccre} overlap a Registry V4 cCRE, of which "
    f"{n_reg_active} is active in a liver biosample; {n_primary} meets the primary (hepatocyte "
    f"ATAC-seq + H3K27ac) criterion -- rs4994910 in both cases.",
    "Whole-liver ATAC-seq (6 ENCODE experiments) is reported for corroboration only and does not "
    "itself define a cCRE (see Supplementary Methods 5.3).",
    "All coordinates are GRCh38. Full methods, LD source reconciliation rules, and the underlying "
    "per-dataset evidence are described in Supplementary Methods 5.3 and implemented in this "
    "notebook.",
    "The lead variant rs11183620 is included for reference (R2=1 with itself); LDlink and Ensembl "
    "both report this exact position under the alternate rsID rs59666428, confirmed via dbSNP "
    "(chr12:46,818,587, GRCh38).",
]})
with pd.ExcelWriter(ST14_PATH, engine="openpyxl") as xw:
    _readme14.to_excel(xw, sheet_name="README", index=False)
    st14.to_excel(xw, sheet_name="LD_cCRE_evidence", index=False)

print(f"{n_r2} variants (of {n_total} tested). Saved: {ST14_PATH}")

## Table S15 — per-dataset and biosample breakdown of chromatin evidence at rs4994910

In [ ]:
# ---- Table S15 -- per-dataset/biosample chromatin evidence at rs4994910 ----
# Builds on `per_dataset` (section 12b, immediately above): adds (a) the ATAC-seq peak
# summit distance to the variant for the PHH panel (narrowPeak format only -- this is the
# specific claim made in the Supplementary Results, "peak summit falling within 60 bp of
# the variant in four [of six]"), and (b) whether each Registry V4 per-biosample call
# falls in the "active" class set already used throughout this notebook.

def _summit_distance(narrowpeak_file, chrom, pos_1based):
    """Distance (bp) from `pos_1based` to the MACS2 summit of the covering peak, or None."""
    with open(narrowpeak_file) as fh:
        for line in fh:
            f = line.rstrip("\n").split("\t")
            if f[0] != f"chr{chrom}":
                continue
            start, end = int(f[1]), int(f[2])
            if start <= (pos_1based - 1) < end:
                summit_1based = start + int(f[9]) + 1
                return abs(summit_1based - pos_1based)
    return None

st15 = per_dataset.copy()
st15["summit_distance_bp"] = pd.NA
st15["within_60bp_of_variant"] = pd.NA

_phh_dir = Path(EVIDENCE_PANELS["PHH"]["dir"])
_is_phh_individual = (st15["panel"] == "PHH") & (st15["evidence_layer"] == "individual sample")
for idx in st15[_is_phh_individual].index:
    dist = _summit_distance(_phh_dir / st15.at[idx, "dataset"],
                             str(short["chrom"].iloc[0]) if "chrom" in short else str(_r["chrom"]),
                             int(st15.at[idx, "pos_grch38"]))
    st15.at[idx, "summit_distance_bp"] = dist
    if dist is not None:
        st15.at[idx, "within_60bp_of_variant"] = dist <= 60

st15["ccre_active_class"] = pd.NA
_is_registry = (st15["evidence_layer"] == "ENCODE Registry V4") & (st15["panel_role"] == "annotation")
st15.loc[_is_registry, "ccre_active_class"] = st15.loc[_is_registry, "call"].isin(CCRE_ACTIVE_CLASSES)

_RENAME15 = {
    "rsid": "rsID", "pos_grch38": "Position (GRCh38)", "evidence_layer": "Evidence layer",
    "panel": "Assay panel", "panel_role": "Panel role", "dataset": "Dataset / file",
    "biosample": "Biosample", "biosample_group": "Biosample category",
    "assay_coverage": "Registry assay coverage", "covers_variant": "Covers variant",
    "interval": "Peak interval (GRCh38)", "interval_bp": "Peak width (bp)",
    "call": "ENCODE Registry V4 class",
    "summit_distance_bp": "Distance: variant to peak summit (bp)",
    "within_60bp_of_variant": "Summit within 60 bp of variant",
    "ccre_active_class": "Registry V4 class is 'active'",
}
_order15 = ["rsid","pos_grch38","evidence_layer","panel","panel_role","dataset",
            "biosample","biosample_group","assay_coverage","covers_variant",
            "interval","interval_bp","summit_distance_bp","within_60bp_of_variant",
            "call","ccre_active_class"]
st15 = st15[_order15].rename(columns=_RENAME15)

_panel_order = {"PHH": 0, "ENCODE_liver": 1, "H3K27ac": 2, "Registry V4": 3}
_layer_order = {"individual sample": 0, "replicated set": 1, "ENCODE Registry V4": 0}
st15 = (st15.assign(_p=st15["Assay panel"].map(_panel_order),
                     _l=st15["Evidence layer"].map(_layer_order))
             .sort_values(["_p", "_l", "Dataset / file"])
             .drop(columns=["_p", "_l"]).reset_index(drop=True))

# quick numeric check against the Supplementary Results claims for rs4994910
_active_col = "Registry V4 class is 'active'"
_phh = st15[(st15["Assay panel"] == "PHH") & (st15["Evidence layer"] == "individual sample")]
_reg = st15[(st15["Evidence layer"] == "ENCODE Registry V4") & (st15["Dataset / file"] != "liver.all (pooled aggregate)")]
_active_cats = sorted(_reg.loc[_reg[_active_col] == True, "Biosample category"].unique())
n_summit60 = int(_phh["Summit within 60 bp of variant"].sum())
n_reg_active = int(_reg[_active_col].sum())
print(f"PHH summit within 60bp: {n_summit60}/{len(_phh)}")
print(f"Registry V4 active:     {n_reg_active}/{len(_reg)}, categories: {_active_cats}")

ST15_PATH = "Table_S15_chromatin_evidence_rs4994910.xlsx"
_readme15 = pd.DataFrame({"Table S15 - Per-dataset and biosample breakdown of "
                          "chromatin accessibility and histone modification evidence for the "
                          "element containing rs4994910": [
    "The regulatory element containing rs4994910 corresponds to ENCODE Registry V4 element "
    "EH38E3013793 (chr12:46,831,106-46,831,375, GRCh38). This table lists, dataset by dataset, "
    "the ATAC-seq and H3K27ac ChIP-seq evidence at this element, plus its ENCODE Registry V4 "
    "cCRE classification.",
    f"Verified against the Supplementary Results: accessible in 6/6 primary hepatocyte (PHH) "
    f"donors, with the ATAC-seq peak summit within 60 bp of the variant in {n_summit60}/6 "
    "(computed directly from the raw MACS2 narrowPeak summit column); accessible in 6/6 "
    "whole-liver ATAC-seq datasets; H3K27ac in 4/4 liver ChIP-seq datasets; pooled Registry V4 "
    f"call dELS; active Registry V4 class in {n_reg_active}/18 individual liver biosamples "
    f"across {len(_active_cats)}/4 biosample categories ({', '.join(_active_cats)}).",
    "PHH ATAC-seq: Cebola lab, 6 donors, MACS2 narrowPeak (summit column used directly for the "
    "summit-distance calculation). Whole-liver ATAC-seq and H3K27ac ChIP-seq: ENCODE portal, "
    "pooled MACS2 peak calls (in-house re-processing; narrowPeak-format .bed for ATAC-seq, "
    "broadPeak for H3K27ac, which does not call a summit). Registry annotation: SCREEN ENCODE "
    "Registry of cCREs V4, per-biosample calls plus the pooled 'liver.all' aggregate track.",
    "'Active' Registry V4 classes: PLS, pELS, dELS, CA-CTCF, CA-TF, CA-only, CA-H3K4me3 (same "
    "definition used throughout this notebook). The pooled liver.all aggregate row is excluded "
    "from the 18-dataset tally.",
    "All coordinates are GRCh38. Full methods are described in Supplementary Methods 5.3 and "
    "implemented in this notebook.",
]})
with pd.ExcelWriter(ST15_PATH, engine="openpyxl") as xw:
    _readme15.to_excel(xw, sheet_name="README", index=False)
    st15.to_excel(xw, sheet_name="Chromatin_evidence", index=False)

st15.to_csv(OUTDIR / "Table_S15_chromatin_evidence_rs4994910.csv", index=False)
print(f"Saved: {ST15_PATH}")
st15.head(10)


## 14. Notes

**Interpreting the output.** The shortlist in section 10 contains every proxy satisfying all
three criteria; it is not filtered to a pre-specified candidate. Variants excluded by the
liver criterion alone are written to `SLC38A4_cCRE_notLiverActive.bed` and listed in the
output, so the effect of that filter can be inspected rather than taken on trust.

**Things worth checking in any given run.**

1. **Registry version.** Section 6 prints the `longLabel` of the cCRE track it used and
   records it in the provenance file. Confirm it is the version you intend before quoting
   any class.
2. **Assay coverage.** cCRE classes are only attainable where the relevant assays were
   performed. A variant reported as inactive in a DNase-only or ChIP-only biosample has not
   been shown to be inactive — section 9 separates these cases and they should not be pooled.
3. **Allele-level matching.** The `allele_match` column records how each proxy was reconciled
   between the two LD services. Rows marked `partial` or `unverified` rest on weaker evidence
   of variant identity than `exact` or `subset`, and are worth inspecting individually.
4. **Panel dependence.** Any proxy close to the R² threshold may be included or excluded
   depending on the reference population. `threshold_is_source_dependent` flags variants where
   the two services disagree; changing `POP` tests sensitivity to the population itself.
5. **Reference assembly.** Everything here is GRCh38. LD is requested with
   `genome_build=grch38` and the cCRE files are hg38, so no coordinate conversion is applied
   at any point.

**Reproducing this.** Requires an LDlink API token (free, https://ldlink.nih.gov/?tab=apiaccess),
`bedtools` on `PATH` (or `BEDTOOLS_BIN` set in section 1), and the per-biosample cCRE files
described in `liver_cCREs/README.md`. The provenance file
written by section 13 records the LD sources, Registry track label, biosample list,
thresholds and package versions used.
